# Dual-pass QC — SD01923_BI

Generated from `(RK)QC_per_sample_dualpass.ipynb`, which was written against SD03522_BG. Paths, figure names and titles now point at SD01923_BI.

Two things still need a human before this runs end to end: the smear cluster ids in the *Removing smear/debris* section, and the domains the Novae QC table flags. Both were chosen for SD03522_BG and mean nothing here until you look at the maps.


In [ ]:
# Imports. scanpy and squidpy do the analysis, the scipy/sklearn bits are
# for the marker statistics and cluster agreement checks further down.

import os
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from scipy.stats import fisher_exact, mannwhitneyu, spearmanr
import seaborn as sns
from sklearn.metrics import cohen_kappa_score, confusion_matrix
import squidpy as sq

# Housekeeping

In [ ]:
# Where figures go and how sharp they are. FIGDIR is also handed to scanpy
# so its own sc.pl.* calls save next to the hand-rolled matplotlib ones.

sc.settings.verbosity = 2
import os as _os_fig
FIGDIR = Path(_os_fig.environ.get('DUALPASS_FIGDIR', '/oak/stanford/scg/lab_mpsnyder/johnck/Projects/RK/Spatial/Novae_persample_INDEPENDENT_FINAL/per_sample/'))
FIGDIR.mkdir(exist_ok=True)
sc.settings.figdir = FIGDIR
plt.rcParams['figure.dpi'] = 300


Panel: ['ABCB1', 'ABCB11', 'ABCC9', 'ABI3', 'AC009242.1', 'AC010201.2', 'AC011287.1', 'AC013287.1', 'AC096667.1', 'AC099792.1', 'AC119673.2', 'ADAM28', 'ADAMTS12', 'ADAMTS16', 'ADAMTS3', 'ADGRG7', 'ADGRV1', 'ADPGK', 'ADRA1A', 'ADRA1B', 'AGPAT3', 'AJ009632.2', 'AL136456.1', 'ALDH3B1', 'ALK', 'ALOX5', 'ALOX5AP', 'ANGPT1', 'ANK1', 'ANKRD18A', 'ANO3', 'ANXA11', 'APOBR', 'APOC1', 'APOC2', 'APOE', 'APP', 'AQP4', 'AQP4-AS1', 'ARHGAP25', 'ARL11', 'ARRB2', 'ASCL2', 'ATP10A', 'ATP2C2', 'ATXN2', 'B3GAT2', 'BAG3', 'BCAT1', 'BCL2A1', 'BCL6', 'BNC2', 'BRINP3', 'BTBD11', 'BTK', 'C1QC', 'C1QL3', 'C22orf39', 'C3', 'C3AR1', 'C9orf72', 'CA10', 'CABP1', 'CADPS2', 'CALCRL', 'CAPG', 'CAPN3', 'CASP8', 'CAV1', 'CBLN2', 'CCDC146', 'CCK', 'CCL4', 'CCRL2', 'CD163', 'CD300A', 'CD300LF', 'CD36', 'CD37', 'CD4', 'CD53', 'CD68', 'CD84', 'CD86', 'CDCP1', 'CDH1', 'CDH12', 'CDH4', 'CDH6', 'CEBPA', 'CEMIP', 'CEMIP2', 'CE_STMN2', 'CE_UNC13A', 'CFAP410', 'CHI3L1', 'CHIT1', 'CHRM3', 'CHST9', 'CLDN11', 'CLSTN2', 'CNDP1', 'CNTN2', 'CNTNAP3B', 'COBLL1', 'COL12A1', 'COL24A1', 'COL25A1', 'COL5A2', 'CPNE4', 'CREG1', 'CRHBP', 'CRYM', 'CSPG4', 'CTNNA3', 'CTSB', 'CTSC', 'CTSD', 'CTSH', 'CUX2', 'CXCL14', 'CXCL16', 'CYBA', 'DAAM2', 'DCN', 'DDI2', 'DDR2', 'DLEU7', 'DLX6-AS1', 'DNER', 'DOCK2', 'DOK3', 'EFHD1', 'EPAS1', 'EPB41L5', 'EPHA4', 'EPSTI1', 'ERBB3', 'ERICD', 'ERMN', 'EXOG', 'EXT1', 'EYA4', 'FAR2', 'FASLG', 'FBLN1', 'FBP1', 'FCER1G', 'FERMT3', 'FGF13', 'FGFR2', 'FGFR3', 'FILIP1', 'FNIP2', 'FOXP2', 'FRAS1', 'FRMD4B', 'FSTL4', 'FSTL5', 'FTL', 'FUS', 'FYB1', 'GAD1', 'GAD2', 'GADD45G', 'GALNT14', 'GAS2L3', 'GEM', 'GJA1', 'GLIS3', 'GLT8D1', 'GMFG', 'GMIP', 'GNB4', 'GNGT2', 'GOLM1', 'GPC5', 'GPNMB', 'GPSM3', 'GRB2', 'GRIK3', 'GRIN2C', 'GRK7', 'GRM8', 'GSTM5', 'HAVCR2', 'HFE', 'HHATL', 'HNRNPA1', 'HOMER2', 'HPGDS', 'HS3ST2', 'HS3ST4', 'HTR2A', 'ID2-AS1', 'IFITM3', 'IGFBP4', 'IGSF1', 'IKZF1', 'IL17D', 'IL17RD', 'IL1A', 'IL1RAPL2', 'ITGA8', 'ITGAM', 'ITGAX', 'KANK1', 'KCNC2', 'KCNE3', 'KCNH5', 'KCNQ2', 'KCNT2', 'KIAA1217', 'KIF22', 'KIF26B', 'KIF4A', 'KIF5A', 'KIT', 'KLK6', 'KNTC1', 'LAMP5', 'LAPTM5', 'LCP2', 'LDB2', 'LGALS9', 'LHX6', 'LILRA2', 'LINC01322', 'LINC02099', 'LINC02232', 'LRMP', 'LRP8', 'LRRK1', 'LRRK2', 'LSP1', 'LY86', 'LYN', 'LYPD6', 'LYPD6B', 'LYVE1', 'MAG', 'MAL', 'MBP', 'MCTP2', 'MECOM', 'MEGF11', 'MEIS2', 'MEPE', 'MFNG', 'MICAL1', 'MID1', 'MILR1', 'MIR137HG', 'MIR497HG', 'MNDA', 'MNX1', 'MOBP', 'MOG', 'MPP1', 'MSR1', 'MX2', 'MYB', 'MYO16', 'MYO1F', 'MYO5B', 'MYRF', 'NCF2', 'NCKAP1L', 'NCSTN', 'NDST4', 'NEK1', 'NELL2', 'NES', 'NIPA1', 'NOS1', 'NOTCH1', 'NPFFR2', 'NPNT', 'NPSR1-AS1', 'NPY1R', 'NR2F2', 'NR2F2-AS1', 'NRP1', 'NTNG1', 'NTNG2', 'NUCKS1', 'NWD2', 'NXPH1', 'NXPH2', 'OBI1-AS1', 'OLFM3', 'OLIG1', 'OLIG2', 'OLR1', 'OPALIN', 'OPTN', 'OSBPL2', 'OSCAR', 'OTOGL', 'P2RY12', 'P2RY13', 'PARP15', 'PARP4', 'PAX6', 'PCDH11X', 'PCSK1', 'PCSK6', 'PDE3B', 'PDGFD', 'PDGFRA', 'PDZRN4', 'PECAM1', 'PFN1', 'PHGDH', 'PHLDB2', 'PIK3AP1', 'PIK3CG', 'PILRA', 'PLA2G7', 'PLAU', 'PLCE1', 'PLCG2', 'PLCH1', 'PLD5', 'PLEKHH1', 'PLEKHH2', 'PM20D1', 'PON3', 'POSTN', 'POU6F2', 'PRDM5', 'PROX1', 'PSEN1', 'PTCHD4', 'PTGS1', 'PTPN6', 'PTPRC', 'PTPRK', 'PTPRZ1', 'PVALB', 'PYCARD', 'RAB42', 'RASGRP1', 'RBFOX3', 'RBM20', 'RCSD1', 'RFTN1', 'RGS10', 'RGS12', 'RGS16', 'RHPN1', 'RIPK3', 'RIT2', 'RNASET2', 'RNF144B', 'RNF220', 'RORB', 'ROS1', 'RSPO2', 'RUBCNL', 'RXFP1', 'RYR3', 'S100Z', 'SAMD5', 'SAMD9', 'SAMSN1', 'SARDH', 'SASH3', 'SCFD1', 'SCN1A-AS1', 'SCUBE1', 'SCUBE3', 'SDK1', 'SEMA3E', 'SEMA5A', 'SEPHS2', 'SERPINA3', 'SERPINH1', 'SFRP2', 'SFTA3', 'SIGLEC7', 'SIGLEC8', 'SLC15A3', 'SLC17A6', 'SLC17A7', 'SLC1A2', 'SLC24A3', 'SLC25A33', 'SLC25A48', 'SLC26A4', 'SLC29A1', 'SLC37A2', 'SLC41A1', 'SLC43A3', 'SLC45A3', 'SLC46A2', 'SLC46A3', 'SLC7A7', 'SLCO2B1', 'SLIT3', 'SMCR8', 'SNCG', 'SNTB2', 'SOAT1', 'SOD1', 'SORCS1', 'SOX10', 'SOX9', 'SPHKAP', 'SPI1', 'SPN', 'SPON1', 'SST', 'ST18', 'ST8SIA4', 'STAC3', 'STAT3', 'STAT5A', 'STK32B', 'STMN2', 'STX3', 'STXBP2', 'SULF1', 'SV2C', 'SYK', 'SYNPR', 'TAC1', 'TACR1', 'TARDBP', 'TBK1', 'TENM1', 'TESPA1', 'TF', 'TFEC', 'TGFB1', 'TGFB2', 'THBS1', 'THEMIS', 'THSD4', 'THSD7B', 'TJP2', 'TLR6', 'TMEM132C', 'TMEM156', 'TMEM74', 'TOB2', 'TOM1L1', 'TP53', 'TPH2', 'TRAF3IP3', 'TREM2', 'TRHDE', 'TRIL', 'TRIM38', 'TRPC5', 'TRPC6', 'TSHZ2', 'TTN', 'TTYH1', 'TUBA4A', 'TXK', 'TYROBP', 'UGT8', 'UNC13A', 'UNC13C', 'UNC5B', 'UNC93B1', 'VAV2', 'VCP', 'VIP', 'VWC2', 'VWC2L', 'VWF', 'WDR49', 'WIF1', 'WWC3', 'ZBBX', 'ZDHHC23', 'ZNF804A', 'ZNF804B']

## Rudimentary manual annotation - first pass

In [ ]:
# Marker dictionary for the manual annotation. Each gene maps to one cell
# type and carries the reason it is there, so the choices stay auditable.

cell_type_markers = {

    # -------------------------------------------------------------------------
    # MOTOR NEURONS
    # Canonical lower motor neuron markers (spinal cord):
    # MNX1 (HB9): transcription factor defining spinal motor neuron identity
    # STMN2: highly enriched in motor neurons; TDP-43-regulated; ALS biomarker
    # CE_STMN2 / CE_UNC13A: custom element (cryptic exon) probes — motor neuron
    # NOS1: nitric oxide synthase 1; enriched in spinal motor neurons
    # CHAT (not in panel but context): replaced here by SLC5A7 (choline transporter)
    # CRYM: mu-crystallin; corticospinal and alpha-motor neuron marker
    # KCNC2: Kv3.2; fast-spiking motor neurons and some interneurons
    # KCNQ2: voltage-gated K+ channel; motor axon; motor neuron identity gene
    # KIF5A: kinesin heavy chain; motor neurons and other neurons; ALS gene
    # SNCG: gamma-synuclein; motor axon marker (spinal ventral horn)
    # ADRA1A / ADRA1B: adrenergic receptors; expressed on spinal motor neurons
    # PRPH (peripheral): not in this panel; closest proxy here is SNCG
    # STAC3: skeletal muscle triad junction; NOT a spinal motor neuron marker
    #        (muscle-specific; included in panel for potential NMJ contamination)
    # FGF13: enriched in neurons including motor neurons
    # -------------------------------------------------------------------------
    "MNX1":     "Motor Neuron",        # Takeuchi 2025: MN_1 SC marker
    "STMN2":    "Motor Neuron",        # TDP-43 target; motor axon enriched
    "CE_STMN2": "Motor Neuron",        # Custom probe: STMN2 cryptic exon (TDP-43 loss marker)
    "CE_UNC13A":"Motor Neuron",        # Custom probe: UNC13A cryptic exon (TDP-43 loss marker)
    "NOS1":     "Motor Neuron",        # Nitric oxide synthase 1; ventral horn MNs
    "CRYM":     "Motor Neuron",        # mu-crystallin; corticospinal/alpha-MN enriched
    "ADRA1A":   "Motor Neuron",        # Alpha-1A adrenoceptor; motor neuron modulation
    "ADRA1B":   "Motor Neuron",        # Alpha-1B adrenoceptor; spinal motor neurons
    "KCNQ2":    "Motor Neuron",        # Kv7.2; motor neuron axon initial segment
    "SNCG":     "Motor Neuron",        # Gamma-synuclein; motor axon marker (Blum 2021)
    "FGF13":    "Motor Neuron",        # FGF homologous factor; neuronal/MN enriched

    # -------------------------------------------------------------------------
    # EXCITATORY INTERNEURONS
    # Spinal cord glutamatergic interneurons: SLC17A6 (VGLUT2) is the canonical
    # marker in spinal cord (SLC17A7/VGLUT1 marks cortex and some DRG).
    # Key TF markers: RORB, PAX6, LDB2
    # -------------------------------------------------------------------------
    "SLC17A6":  "Excitatory Interneuron",  # VGLUT2; dominant glutamatergic marker in SC
    "SLC17A7":  "Excitatory Interneuron",  # VGLUT1; corticospinal axon terminals / dorsal horn
    "RORB":     "Excitatory Interneuron",  # Layer 4 / dorsal horn excitatory TF
    "PAX6":     "Excitatory Interneuron",  # Dorsal excitatory interneuron progenitor TF
    "LDB2":     "Excitatory Interneuron",  # Dorsal excitatory interneuron (Blum 2021)
    "FOXP2":    "Excitatory Interneuron",  # V1/V0 interneuron marker; also Micro_1 (minor)
    "PROX1":    "Excitatory Interneuron",  # dI4/dIL interneuron; inhibitory progenitor (dual role)
    "CBLN2":    "Excitatory Interneuron",  # Cerebellin 2; dorsal horn excitatory synapses
    "CPNE4":    "Excitatory Interneuron",  # Copine 4; spinal excitatory neuron enriched
    "C1QL3":    "Excitatory Interneuron",  # Complement C1q-like 3; synapse organiser, excit. neurons
    "GRIK3":    "Excitatory Interneuron",  # Kainate receptor GluR7; excitatory interneurons
    "GRIN2C":   "Excitatory Interneuron",  # NMDA receptor subunit; SC excitatory neurons
    "CA10":     "Excitatory Interneuron",  # Carbonic anhydrase 10; dorsal horn excitatory neurons
    "CABP1":    "Excitatory Interneuron",  # Calcium-binding protein 1; excitatory CNS neurons
    "CADPS2":   "Excitatory Interneuron",  # Dense-core vesicle exocytosis; excitatory neurons
    "DNER":     "Excitatory Interneuron",  # Delta/notch-like EGF; Purkinje/excitatory neurons
    "EXT1":     "Excitatory Interneuron",  # Heparan sulfate synth; dorsal horn neuron development
    "GRM8":     "Excitatory Interneuron",  # mGluR8; presynaptic modulation, SC excit. neurons
    "HS3ST2":   "Excitatory Interneuron",  # Heparan sulfate 3-O-sulfotransferase; dorsal horn neurons
    "HS3ST4":   "Excitatory Interneuron",  # Dorsal horn excitatory neuron enriched (Blum 2021)
    "NELL2":    "Excitatory Interneuron",  # Neural EGFL-like 2; ventral excitatory interneurons
    "NTNG1":    "Excitatory Interneuron",  # Netrin-G1; excitatory synapse formation, SC neurons
    "NTNG2":    "Excitatory Interneuron",  # Netrin-G2; dorsal horn excitatory synapses
    "OLFM3":    "Excitatory Interneuron",  # Olfactomedin 3; dorsal horn excitatory neurons
    "RYR3":     "Excitatory Interneuron",  # Ryanodine receptor 3; SC excitatory neurons
    "SPON1":    "Excitatory Interneuron",  # Spondin 1; floor plate / excitatory SC neurons
    "SDK1":     "Excitatory Interneuron",  # Sidekick 1; dorsal horn laminae II excitatory neurons
    "SORCS1":   "Excitatory Interneuron",  # SorCS1; neuronal vesicle trafficking; SC excit.
    "SV2C":     "Excitatory Interneuron",  # Synaptic vesicle glycoprotein 2C; SC neurons
    "VWC2":     "Excitatory Interneuron",  # von Willebrand factor C domain 2; SC interneuron
    "VWC2L":    "Excitatory Interneuron",  # VWC2-like; dorsal horn excitatory neurons
    "SYNPR":    "Excitatory Interneuron",  # Synaptoporin; dorsal horn excitatory synapses
    "HOMER2":   "Excitatory Interneuron",  # Postsynaptic density protein; excitatory neurons
    "CDH12":    "Excitatory Interneuron",  # Cadherin 12; dorsal horn neuron subtype identity
    "CDH4":     "Excitatory Interneuron",  # Cadherin 4; SC dorsal horn excitatory neurons
    "CDH6":     "Excitatory Interneuron",  # Cadherin 6; dorsal horn laminar organisation
    "CRHBP":    "Excitatory Interneuron",  # CRH-binding protein; dorsal horn peptidergic neurons
    "CCK":      "Excitatory Interneuron",  # Cholecystokinin; dorsal horn excitatory interneurons
    "CHRM3":    "Excitatory Interneuron",  # Muscarinic M3; SC excitatory interneurons
    "HTR2A":    "Excitatory Interneuron",  # Serotonin 2A receptor; dorsal horn excit. neurons
    "LYPD6":    "Excitatory Interneuron",  # LY6/PLAUR domain 6; dorsal horn neuron subtype
    "LYPD6B":   "Excitatory Interneuron",  # LY6/PLAUR domain 6B; dorsal horn neurons
    "NPY1R":    "Excitatory Interneuron",  # NPY receptor Y1; dorsal horn neurons
    "PCSK1":    "Excitatory Interneuron",  # Proprotein convertase 1; neuropeptide processing neurons
    "PCSK6":    "Excitatory Interneuron",  # Proprotein convertase 6; SC neurons
    "PDZRN4":   "Excitatory Interneuron",  # PDZ domain ring finger 4; SC excitatory neurons
    "PLEKHH1":  "Excitatory Interneuron",  # Pleckstrin homology domain; SC excit. neurons
    "PLEKHH2":  "Excitatory Interneuron",  # SC dorsal horn excitatory neurons
    "POU6F2":   "Excitatory Interneuron",  # POU TF; dorsal horn dI3 excitatory interneurons
    "TACR1":    "Excitatory Interneuron",  # Substance P receptor NK1R; dorsal horn excit. neurons
    "TAC1":     "Excitatory Interneuron",  # Substance P/tachykinin; dorsal horn excit. peptidergic
    "TRPC5":    "Excitatory Interneuron",  # TRP channel 5; dorsal horn pain-processing neurons
    "TRPC6":    "Excitatory Interneuron",  # TRP channel 6; SC sensory/excit. interneurons
    "UNC13C":   "Excitatory Interneuron",  # Munc13-3; SC excitatory synapse release machinery
    "WIF1":     "Excitatory Interneuron",  # WNT inhibitory factor 1; dorsal horn neurons (Yadav 2023)
    "TSHZ2":    "Excitatory Interneuron",  # Teashirt zinc finger 2; V1-related SC neurons
    "PTPRC":    "Broad/Multiple",          # CD45 — primarily immune but sometimes annotated here

    # -------------------------------------------------------------------------
    # INHIBITORY INTERNEURONS
    # Spinal GABAergic and glycinergic interneurons: GAD1, GAD2, SLC6A5
    # Canonical inhibitory TFs: LHX6, DLX, MEIS2
    # Renshaw cells: KCNC2-high, parvalbumin+
    # -------------------------------------------------------------------------
    "GAD1":     "Inhibitory Interneuron",  # Glutamate decarboxylase 1; pan-GABAergic marker
    "GAD2":     "Inhibitory Interneuron",  # Glutamate decarboxylase 2; pan-GABAergic marker
    "LHX6":     "Inhibitory Interneuron",  # LIM homeobox 6; MGE-derived inhibitory neurons
    "PVALB":    "Inhibitory Interneuron",  # Parvalbumin; Renshaw cells / fast inhibitory INs
    "SST":      "Inhibitory Interneuron",  # Somatostatin; inhibitory interneurons (dorsal horn)
    "VIP":      "Inhibitory Interneuron",  # Vasoactive intestinal peptide; inhibitory INs
    "NXPH1":    "Inhibitory Interneuron",  # Neurexophilin 1; inhibitory synapse; glycinergic SC INs
    "NXPH2":    "Inhibitory Interneuron",  # Neurexophilin 2; inhibitory interneurons
    "LAMP5":    "Inhibitory Interneuron",  # Lysosomal-associated protein 5; CGE-derived inhib. INs
    "BRINP3":   "Inhibitory Interneuron",  # BMP/RA-inducible neural protein 3; spinal inhib. INs
    "DLX6-AS1": "Inhibitory Interneuron",  # DLX6 antisense; pan-inhibitory neuron locus marker
    "MEIS2":    "Inhibitory Interneuron",  # MEIS2 TF; MGE-derived inhibitory neuron identity
    "ANO3":     "Inhibitory Interneuron",  # Anoctamin 3; spinal inhibitory interneurons
    "NUCKS1":   "Inhibitory Interneuron",  # Nuclear casein kinase substrate; inhib. neuron enriched
    "KCNC2":    "Inhibitory Interneuron",  # Kv3.2; fast-spiking inhibitory (Renshaw) neurons
    "KCNH5":    "Inhibitory Interneuron",  # Kv10.2 (EAG2); inhibitory interneuron subtype
    "KCNT2":    "Inhibitory Interneuron",  # Slack-B K+ channel; SC inhibitory neurons
    "LRMP":     "Inhibitory Interneuron",  # Lung resistance-related; SC inhibitory INs (Blum 2021)
    "NPFFR2":   "Inhibitory Interneuron",  # Neuropeptide FF receptor 2; SC pain-modulating inhib INs
    "TPH2":     "Inhibitory Interneuron",  # Tryptophan hydroxylase 2; serotonergic (raphe desc. axons)
    "STK32B":   "Inhibitory Interneuron",  # Serine-threonine kinase 32B; spinal inhibitory neurons
    "GLIS3":    "Inhibitory Interneuron",  # GLIS3 TF; spinal inhibitory interneuron subtype
    "CNTN2":    "Inhibitory Interneuron",  # Contactin 2; dorsal root entry zone / spinal INs

    # -------------------------------------------------------------------------
    # ASTROCYTES
    # Spinal cord: fibrillary (WM, CD44-hi/GFAP-hi) vs protoplasmic (GM, SLC1A2-hi)
    # Takeuchi 2025: Astro_1 (CD44, CCDC85A = WM), Astro_2 (SLC1A2, TENM2 = GM),
    # Astro_3 (GABRB1, GRID2 = GM, partial WM)
    # -------------------------------------------------------------------------
    "AQP4":     "Astrocyte",    # Aquaporin 4; pan-astrocyte canonical marker
    "AQP4-AS1": "Astrocyte",    # AQP4 antisense lncRNA; astrocyte-enriched locus
    "SLC1A2":   "Astrocyte",    # EAAT2 glutamate transporter; protoplasmic astrocyte
    "GJA1":     "Astrocyte",    # Connexin 43; astrocyte gap junctions
    "SOX9":     "Astrocyte",    # SOX9 TF; pan-astrocyte (also OPC at lower level)
    "PHGDH":    "Astrocyte",    # Phosphoglycerate dehydrogenase; astrocyte serine synthesis
    "ALDH3B1":  "Astrocyte",    # Aldehyde dehydrogenase 3B1; astrocyte-enriched enzyme
    "CHI3L1":   "Astrocyte",    # YKL-40; reactive/fibrillary astrocyte marker
    "SERPINA3":  "Astrocyte",   # Alpha-1-antichymotrypsin; reactive astrocyte marker in ALS
    "C3":       "Astrocyte",    # Complement C3; A1 reactive astrocyte marker
    "TTYH1":    "Astrocyte",    # Tweety homolog 1; astrocyte-enriched chloride channel
    "GPC5":     "Astrocyte",    # Glypican 5; astrocyte proteoglycan (high in SC astrocytes)
    "FGFR3":    "Astrocyte",    # FGF receptor 3; astrocyte identity (Blum 2021)
    "THSD7B":   "Astrocyte",    # Thrombospondin type-1 domain 7B; SC astrocyte enriched
    "CLDN11":   "Oligodendrocyte",  # NOTE: annotated below — also expressed by astrocytes but
                                    # primary marker is oligodendrocyte (see Oligodendrocyte block)
    "TGFB2":    "Astrocyte",    # TGF-beta 2; astrocyte secreted; reactive astrocyte upregulated
    "STAT3":    "Astrocyte",    # STAT3; reactive astrocyte signalling TF (also broad)

    # -------------------------------------------------------------------------
    # OLIGODENDROCYTES (mature myelinating)
    # Canonical: MBP, MOG, MOBP, MAG, MAL, PLP1, CNP, CLDN11, UGT8, MYRF, SOX10
    # Takeuchi 2025 SC subtypes: Oligo_1 (CCSER1/CACNA1B), Oligo_2 (GNA14/COL18A1),
    # Oligo_3 (NLGN1/PLXDC2), Oligo_4 (SGCZ/KCNMB4 — ALS-reduced)
    # -------------------------------------------------------------------------
    "MBP":      "Oligodendrocyte",  # Myelin basic protein; canonical myelin marker
    "MOG":      "Oligodendrocyte",  # Myelin oligodendrocyte glycoprotein; outer myelin
    "MOBP":     "Oligodendrocyte",  # Myelin-associated oligodendrocyte basic protein
    "MAG":      "Oligodendrocyte",  # Myelin-associated glycoprotein; periaxonal myelin
    "MAL":      "Oligodendrocyte",  # Myelin and lymphocyte protein; compact myelin
    "UGT8":     "Oligodendrocyte",  # UDP glycosyltransferase 8; galactocerebroside synthesis
    "MYRF":     "Oligodendrocyte",  # Myelin regulatory factor; mature OL TF
    "SOX10":    "Oligodendrocyte",  # SOX10 TF; pan-oligodendrocyte lineage (OL+OPC)
    "CNDP1":    "Oligodendrocyte",  # Carnosine dipeptidase 1; mature oligodendrocyte
    "OLIG1":    "Oligodendrocyte",  # OLIG1 TF; mature oligodendrocyte (also OPC)
    "OLIG2":    "OPC",              # OLIG2 TF; highest in OPC / immature OL (also mature)
    "ERMN":     "Oligodendrocyte",  # Ermin; myelin maintenance protein
    "OPALIN":   "Oligodendrocyte",  # Opalin; SC oligodendrocyte-enriched protein
    "CLDN11":   "Oligodendrocyte",  # Claudin 11; tight junctions in myelin (primary label)
    "ST18":     "Oligodendrocyte",  # ST18 TF; mature oligodendrocyte enriched
    "RNF220":   "Oligodendrocyte",  # Ring finger protein 220; oligodendrocyte differentiation
    "MYRF":     "Oligodendrocyte",  # already above
    "DAAM2":    "Oligodendrocyte",  # Dishevelled-associated activator 2; OL myelination
    "SULF1":    "Oligodendrocyte",  # Sulfatase 1; heparan sulfate remodelling in OLs
    "EPAS1":    "Oligodendrocyte",  # HIF-2alpha; OL and endothelial hypoxia response
    "PLP1":     "Oligodendrocyte",  # NOTE: PLP1 not in panel; CLDN11/MBP are proxies
    "STMN2":    "Motor Neuron",     # (already in MN block; skipping duplicate)
    "SLC45A3":  "Oligodendrocyte",  # Solute carrier 45A3; SC oligodendrocyte-enriched
    "ANXA11":   "Broad/Multiple",   # Annexin A11; ALS gene (aggregates); expressed broadly

    # -------------------------------------------------------------------------
    # OPC (Oligodendrocyte Precursor Cells)
    # Canonical: PDGFRA, CSPG4 (NG2), VCAN, OLIG2
    # -------------------------------------------------------------------------
    "PDGFRA":   "OPC",   # PDGF receptor alpha; canonical OPC marker (Takeuchi 2025 Opc_1)
    "CSPG4":    "OPC",   # NG2 chondroitin sulfate proteoglycan; OPC cell surface marker
    "PTPRZ1":   "OPC",   # PTPRZ1 receptor tyrosine phosphatase; OPC (also astrocyte)
    "PCDH11X":  "OPC",   # Protocadherin 11X; OPC subtype marker (Blum 2021)
    "MEGF11":   "OPC",   # MEGF11; OPC phagocytic receptor
    "LRRK1":    "OPC",   # Leucine-rich repeat kinase 1; OPC enriched (Allen Brain)

    # -------------------------------------------------------------------------
    # MICROGLIA
    # Canonical: P2RY12, P2RY13, TMEM119, CX3CR1, IBA1/AIF1, TREM2, TYROBP
    # Takeuchi 2025 SC: Micro_1 (KCNIP1, FOXP2), Micro_2 (SPP1, IPCEF1),
    # Micro_3 (GPNMB, IQGAP2), Micro_4 (SLC24A2, IL1RAPL1),
    # Micro_5 (GNA13, NAMPT), Micro_6 (FTL, EEF1A1)
    # Disease-associated microglia (DAM): TREM2, SPP1, GPNMB, CD68, LPL, APOE
    # -------------------------------------------------------------------------
    "P2RY12":   "Microglia",  # Homeostatic microglia canonical marker
    "P2RY13":   "Microglia",  # P2Y purinoceptor 13; homeostatic microglia
    "TREM2":    "Microglia",  # Triggering receptor myeloid cells 2; DAM / phagocytic microglia
    "TYROBP":   "Microglia",  # DAP12; microglia/macrophage signalling adaptor
    "CD68":     "Microglia",  # Macrosialin; pan-myeloid, strong in activated microglia
    "SPI1":     "Microglia",  # PU.1 TF; master myeloid regulator; microglia identity
    "ABI3":     "Microglia",  # ABI family member 3; microglia; ALS GWAS hit
    "GPNMB":    "Microglia",  # Glycoprotein NMB; Micro_3 subtype (Takeuchi 2025); DAM
    "C1QC":     "Microglia",  # Complement C1q chain C; microglia synaptic pruning
    "C3AR1":    "Microglia",  # Complement C3a receptor; microglia activation
    "CD86":     "Microglia",  # Co-stimulatory molecule; activated microglia
    "CD84":     "Microglia",  # SLAM family; microglia/macrophage
    "ITGAM":    "Microglia",  # CD11b; myeloid/microglia integrin
    "ITGAX":    "Microglia",  # CD11c; activated microglia and dendritic cells
    "LAPTM5":   "Microglia",  # Lysosomal protein transmembrane 5; microglia
    "LY86":     "Microglia",  # MD-1; LPS-signalling co-receptor; microglia
    "LYVE1":    "Microglia",  # LYVE1; perivascular macrophages and some microglia
    "FTL":      "Microglia",  # Ferritin light chain; Micro_6 (Takeuchi 2025); also astrocyte
    "FCER1G":   "Microglia",  # Fc epsilon receptor gamma; microglia / macrophages
    "DOCK2":    "Microglia",  # Dedicator of cytokinesis 2; microglia migration
    "HAVCR2":   "Microglia",  # TIM-3; exhausted microglia / disease-associated
    "PILRA":    "Microglia",  # Paired Ig-like receptor A; microglia / macrophages
    "RGS10":    "Microglia",  # Regulator of G-protein 10; homeostatic microglia
    "SIGLEC7":  "Microglia",  # Sialic acid-binding Ig-like lectin 7; microglia (NK cells also)
    "SIGLEC8":  "Microglia",  # Siglec-8; myeloid / eosinophils / microglia subtype
    "TRIL":     "Microglia",  # TLR4 interactor with LRR; microglia innate immune
    "TLR6":     "Microglia",  # Toll-like receptor 6; microglia innate immune signalling
    "TYROBP":   "Microglia",  # already listed; duplicate key removed in final use
    "SLC37A2":  "Microglia",  # Glucose-6-phosphate transporter; microglia
    "SLC43A3":  "Microglia",  # SLC43A3; microglia-enriched transporter
    "CTSD":     "Microglia",  # Cathepsin D; lysosomal; microglia/macrophage
    "CTSB":     "Microglia",  # Cathepsin B; microglia phagolysosomes
    "CTSC":     "Microglia",  # Cathepsin C; myeloid cells including microglia
    "CTSH":     "Microglia",  # Cathepsin H; microglia enriched
    "RGS16":    "Microglia",  # RGS16; microglia / macrophage subtype
    "PYCARD":   "Microglia",  # ASC (apoptosis-associated speck); NLRP3 inflammasome; microglia
    "MNDA":     "Microglia",  # Myeloid nuclear differentiation antigen; myeloid/microglia
    "GMFG":     "Microglia",  # Glia maturation factor gamma; microglia motility
    "SAMSN1":   "Microglia",  # SAM domain/SH3 domain; microglia signalling
    "RNASET2":  "Microglia",  # Ribonuclease T2; microglia lysosomal enzyme
    "OLR1":     "Microglia",  # Oxidised LDL receptor 1; activated microglia / macrophage
    "RFTN1":    "Microglia",  # Raftlin; microglia/macrophage lipid raft
    "CD300A":   "Microglia",  # CD300a inhibitory receptor; microglia
    "CD300LF":  "Microglia",  # CD300lf; microglia inhibitory receptor
    "CAPG":     "Microglia",  # Macrophage capping protein; myeloid cells
    "MYO1F":    "Microglia",  # Myosin IF; microglia migration
    "CXCL16":   "Microglia",  # Chemokine; microglia/macrophage scavenger receptor
    "LGALS9":   "Microglia",  # Galectin-9; microglia immunomodulation (also astrocytes)
    "HPGDS":    "Microglia",  # Hematopoietic prostaglandin D synthase; microglia subtype
    "LRMP":     "Inhibitory Interneuron",  # reassigned above; avoid duplicate — see note below
    "PIK3CG":   "Microglia",  # PI3K gamma; myeloid cells including microglia
    "PIK3AP1":  "Microglia",  # PI3K adaptor 1; B cells and microglia
    "PLCG2":    "Microglia",  # PLCgamma2; microglia; ALS/AD risk gene
    "SYK":      "Microglia",  # Spleen tyrosine kinase; microglia / macrophages
    "LYN":      "Microglia",  # Lyn kinase; microglia signalling
    "FYB1":     "Microglia",  # FYN-binding protein 1; microglia/T cell
    "RASGRP1":  "Microglia",  # RAS guanyl nucleotide-releasing protein 1; myeloid / T cells
    "IKZF1":    "Microglia",  # Ikaros TF; myeloid/lymphoid; microglia precursors
    "CYBA":     "Microglia",  # p22-phox; NADPH oxidase; microglia/macrophage
    "NCF2":     "Microglia",  # p67-phox; NADPH oxidase; myeloid cells
    "NCKAP1L":  "Microglia",  # HEM1; WAVE complex; microglia morphology
    "DOCK3":    "Microglia",  # NOTE: DOCK3 not in panel; DOCK2 annotated above
    "APOBR":    "Microglia",  # ApoB receptor; microglia/macrophage lipid metabolism
    "APOC1":    "Microglia",  # Apolipoprotein C1; microglia/macrophage lipid
    "APOC2":    "Microglia",  # Apolipoprotein C2; microglia/macrophage
    "APOE":     "Microglia",  # Apolipoprotein E; microglia-dominant in CNS (also astrocyte)
    "MYB":      "Microglia",  # c-Myb TF; microglia maintenance TF (Hoeffel 2018)
    "PARP15":   "Microglia",  # Poly-ADP-ribose polymerase 15; myeloid cells
    "EPSTI1":   "Microglia",  # Epithelial stromal interaction 1; microglia/macrophage
    "ARHGAP25": "Microglia",  # Rho GTPase-activating protein 25; microglia motility
    "STXBP2":   "Microglia",  # STXBP2; degranulation; microglia/NK/macrophage
    "CCRL2":    "Microglia",  # Chemokine receptor-like 2; microglia/macrophage
    "OSCAR":    "Microglia",  # Osteoclast-associated receptor; myeloid cells / microglia
    "PLA2G7":   "Microglia",  # Platelet-activating factor acetylhydrolase; macrophage/microglia
    "GNGT2":    "Microglia",  # G protein gamma subunit 2; microglia
    "RGS12":    "Microglia",  # RGS12; myeloid / microglia signalling

    # -------------------------------------------------------------------------
    # MONOCYTE / MACROPHAGE
    # CNS-border macrophages (perivascular, meningeal, choroid), infiltrating
    # monocytes in ALS tissue; Takeuchi 2025 SC: Macro_1 (MRC1, CD163)
    # -------------------------------------------------------------------------
    "CD163":    "Monocyte/Macrophage",  # Hemoglobin scavenger; M2/perivascular macrophage
    "MSR1":     "Monocyte/Macrophage",  # Scavenger receptor A; macrophage
    "CHIT1":    "Monocyte/Macrophage",  # Chitotriosidase; monocyte-derived macrophage marker
    "CD36":     "Monocyte/Macrophage",  # Scavenger receptor; monocyte/macrophage lipid uptake
    "CEBPA":    "Monocyte/Macrophage",  # CEBP-alpha TF; myeloid differentiation TF
    "ALOX5":    "Monocyte/Macrophage",  # 5-lipoxygenase; myeloid / macrophage leukotriene pathway
    "ALOX5AP":  "Monocyte/Macrophage",  # FLAP; 5-LOX activating protein; macrophage
    "MX2":      "Monocyte/Macrophage",  # MX2 IFN-stimulated gene; macrophage / monocyte
    "BCL2A1":   "Monocyte/Macrophage",  # Bcl-2 A1; myeloid survival; macrophage
    "PLAU":     "Monocyte/Macrophage",  # Urokinase plasminogen activator; macrophage
    "ADAM28":   "Monocyte/Macrophage",  # ADAM metalloprotease 28; macrophage / NK
    "LILRA2":   "Monocyte/Macrophage",  # Leukocyte Ig-like receptor A2; monocyte/macrophage
    "FCGR":     "Monocyte/Macrophage",  # (not in panel as-is; placeholder)
    "SLC46A2":  "Monocyte/Macrophage",  # Folate transporter; macrophage enriched
    "SLC7A7":   "Monocyte/Macrophage",  # y+L amino acid transporter; macrophage/monocyte
    "SOAT1":    "Monocyte/Macrophage",  # Sterol O-acyltransferase 1; lipid-laden macrophage
    "TGFB1":    "Monocyte/Macrophage",  # TGF-beta 1; macrophage-secreted (also astrocyte)
    "TFEC":     "Monocyte/Macrophage",  # Transcription factor EC; macrophage lysosome TF
    "ANK1":     "Monocyte/Macrophage",  # Ankyrin 1; erythroid + macrophage expression
    "SAMD9":    "Monocyte/Macrophage",  # Sterile alpha motif domain 9; macrophage IFN response

    # -------------------------------------------------------------------------
    # ENDOTHELIAL CELLS
    # CNS vasculature; Takeuchi 2025 SC: Endothelial_1 (FLT1, VWF)
    # Also includes ABCB1 (blood-brain barrier P-gp) and PECAM1 (CD31)
    # -------------------------------------------------------------------------
    "VWF":      "Endothelial",   # von Willebrand factor; pan-endothelial canonical marker
    "PECAM1":   "Endothelial",   # CD31; pan-endothelial adhesion molecule
    "ABCB1":    "Endothelial",   # MDR1/P-glycoprotein; blood-brain barrier endothelium
    "CALCRL":   "Endothelial",   # CGRP receptor component; CNS endothelium
    "EPAS1":    "Endothelial",   # HIF-2alpha; strong in CNS endothelium (also OL, see above)
    "NRP1":     "Endothelial",   # Neuropilin 1; endothelial tip cells / angiogenesis
    "ANGPT1":   "Endothelial",   # Angiopoietin 1; vascular stabilisation; also pericyte
    "UNC5B":    "Endothelial",   # UNC5B netrin receptor; CNS arterial endothelium
    "SLCO2B1":  "Endothelial",   # Organic anion transporter; BBB endothelium
    "SLC29A1":  "Endothelial",   # ENT1 adenosine transporter; endothelium
    "COBLL1":   "Endothelial",   # Cordon-bleu WH2 domain; endothelial cell identity
    "MECOM":    "Endothelial",   # EVI1 TF; CNS endothelial / haematopoietic progenitors
    "THSD4":    "Endothelial",   # Thrombospondin type-1 4; endothelium; also MN marker (EYA4 context)
    "NOTCH1":   "Endothelial",   # NOTCH1; arterial endothelium; also NSC (context-dependent)
    "EPAS1":    "Endothelial",   # see note above
    "APOLD1":   "Endothelial",   # (not in panel; placeholder)
    "CLDN11":   "Oligodendrocyte",  # final assignment: Oligodendrocyte (see above)
    "TJP2":     "Endothelial",   # Tight junction protein 2; BBB tight junctions
    "CEMIP2":   "Endothelial",   # Cell migration-inducing protein 2; CNS endothelium
    "EFHD1":    "Endothelial",   # EF-hand domain 1; SC endothelial enriched

    # -------------------------------------------------------------------------
    # PERICYTES
    # CNS mural cells; Takeuchi 2025 SC: Pericyte_1 (PDGFRB, DLC1)
    # -------------------------------------------------------------------------
    "ABCC9":    "Pericyte",   # SUR2 (KATP channel); canonical CNS pericyte marker
    "KCNJ8":    "Pericyte",   # Kir6.1; CNS pericyte (not in panel; context note)
    "PDGFD":    "Pericyte",   # PDGF-D; pericyte-secreted ligand
    "GJA1":     "Astrocyte",  # Already assigned astrocyte (final label astrocyte)
    "RGS5":     "Pericyte",   # (not in panel; reference)
    "ANGPT1":   "Pericyte",   # Angiopoietin-1; pericyte-secreted (also endothelial above)
    "MCTP2":    "Pericyte",   # Multiple C2-domain transmembrane protein 2; pericyte enriched

    # -------------------------------------------------------------------------
    # FIBROBLASTS (perivascular and meningeal)
    # Takeuchi 2025 SC: Meninges_2 (DCN, ABCA8); PeriVascular_1 (LAMA2, ABCA9)
    # -------------------------------------------------------------------------
    "DCN":      "Fibroblast",   # Decorin; canonical fibroblast ECM marker
    "COL12A1":  "Fibroblast",   # Collagen XII alpha 1; fibroblast/stromal ECM
    "COL5A2":   "Fibroblast",   # Collagen V alpha 2; fibroblast ECM
    "COL24A1":  "Fibroblast",   # Collagen XXIV; meningeal fibroblast enriched
    "COL25A1":  "Fibroblast",   # Collagen XXV; CNS fibroblast/meningeal
    "FBLN1":    "Fibroblast",   # Fibulin 1; fibroblast/ECM matrix
    "SFRP2":    "Fibroblast",   # Secreted frizzled-related 2; fibroblast WNT modulation
    "POSTN":    "Fibroblast",   # Periostin; fibroblast ECM/periosteum
    "CEMIP":    "Fibroblast",   # KIAA1199; fibroblast/stromal matrix
    "DDR2":     "Fibroblast",   # Discoidin domain receptor 2; fibroblast collagen sensor
    "SERPINH1": "Fibroblast",   # HSP47; collagen chaperone; fibroblast
    "THBS1":    "Fibroblast",   # Thrombospondin 1; fibroblast/astrocyte ECM (primary fibroblast)
    "ADAMTS3":  "Fibroblast",   # ADAMTS3; collagen propeptidase; fibroblast
    "ADAMTS12": "Fibroblast",   # ADAMTS12; ECM remodelling; fibroblast enriched
    "ADAMTS16": "Fibroblast",   # ADAMTS16; ECM; meningeal fibroblast
    "FRAS1":    "Fibroblast",   # Fraser syndrome 1; basement membrane; fibroblast
    "PRDM5":    "Fibroblast",   # PRDM5 TF; fibroblast collagen gene regulation
    "ITGA8":    "Fibroblast",   # Integrin alpha 8; fibroblast ECM adhesion

    # -------------------------------------------------------------------------
    # VLMC (Vascular Leptomeningeal Cells / meningeal fibroblasts type 1)
    # Takeuchi 2025 SC: Meninges_1 (SLC4A4); distinguished from perivascular fibroblasts
    # VLMC express COLEC12, LAMA1, and high NR2F2 in CNS atlases (Zeisel 2018)
    # -------------------------------------------------------------------------
    "NR2F2":    "VLMC",    # NR2F2 (COUP-TFII); canonical VLMC/leptomeningeal marker
    "NR2F2-AS1":"VLMC",    # NR2F2 antisense; leptomeningeal/VLMC locus
    "SLIT3":    "VLMC",    # Slit3; meningeal fibroblast / VLMC enriched
    "LAMA2":    "Fibroblast",  # Laminin alpha 2; perivascular fibroblast (Takeuchi 2025)
    "RSPO2":    "VLMC",    # R-spondin 2; meningeal/VLMC WNT amplifier

    # -------------------------------------------------------------------------
    # EPENDYMAL CELLS
    # Takeuchi 2025 SC: Ependymal_1 (DNAH11, CFAP44); central canal lining
    # -------------------------------------------------------------------------
    "CFAP410":  "Ependymal",  # Cilia and flagella associated 410; ependymal cilia
    "DNAI1":    "Ependymal",  # (not in panel; reference)
    "TUBA4A":   "Broad/Multiple",  # Tubulin alpha 4A; pan-neuronal/ALS gene (TDP-43 target); not ependymal-specific

    # -------------------------------------------------------------------------
    # SCHWANN CELLS (peripheral; may appear at SC roots / post-mortem artefact)
    # -------------------------------------------------------------------------
    "MPP1":     "Schwann",   # Membrane protein palmitoylated 1; Schwann cell enriched

    # -------------------------------------------------------------------------
    # T CELLS / NK CELLS
    # Takeuchi 2025 SC: Lymphocyte_1 (CD96, ITK)
    # -------------------------------------------------------------------------
    "CD4":      "T cell",    # CD4 T helper cell surface marker
    "CD37":     "T cell",    # CD37 tetraspanin; pan-lymphocyte / B cell
    "THEMIS":   "T cell",    # Thymocyte selection-associated; mature T cell marker
    "TXK":      "T cell",    # TXK tyrosine kinase; T cell signalling
    "TESPA1":   "T cell",    # Thymocyte-expressed positive selection-associated 1; T cells
    "TRAF3IP3": "T cell",    # TRAF3-interacting protein 3; T cell
    "SASH3":    "T cell",    # SAM and SH3 domain 3; T cell signalling
    "LSP1":     "T cell",    # Lymphocyte-specific protein 1; T cell / NK
    "LCP2":     "T cell",    # SLP-76; T cell receptor signalling
    "CD53":     "T cell",    # CD53 tetraspanin; pan-leukocyte
    "LRMP":     "T cell",    # Jaw1; lymphoid-restricted; T/B cells (NOTE: same gene as inhibitory neuron entry above — in context of immune cells)
    "BCL6":     "T cell",    # BCL6 TF; germinal centre / T follicular helper cells
    "IL17D":    "T cell",    # IL-17D; T cell and NK subtype cytokine
    "FASLG":    "T cell",    # FasL; cytotoxic T cells and NK cells
    "DLEU7":    "T cell",    # Deleted in lymphocytic leukaemia 7; B/T cell
    "SPN":      "T cell",    # CD43; pan-leukocyte sialophorin
    "PTPN6":    "T cell",    # SHP-1; T/B cell receptor signalling phosphatase
    "SIGLEC7":  "NK cell",   # Siglec-7; NK cell inhibitory receptor (also microglia above — NK primary)
    "ADAM28":   "NK cell",   # ADAM28; NK cell (also macrophage; NK primary in CNS infiltrate)
    "MILR1":    "T cell",    # Mast cell/IL33 receptor-like; also NK/T

    # -------------------------------------------------------------------------
    # ALS GENES
    # Causative or strong GWAS risk genes for ALS; not cell-type-restricted markers.
    # Expression patterns from Takeuchi 2025: ATXN2, FUS, TAF15 highly expressed in
    # microglia and multiple cell types; TBK1, OPTN in many cell types;
    # SOD1 ubiquitous; TARDBP ubiquitous; NEK1 broadly expressed.
    # -------------------------------------------------------------------------
    "FUS":      "ALS gene",    # RNA-binding protein; ALS/FTD causative; ubiquitous expression
    "TARDBP":   "ALS gene",    # TDP-43; ALS/FTD causative; ubiquitous but enriched neurons
    "SOD1":     "ALS gene",    # Superoxide dismutase 1; ALS causative; ubiquitous
    "C9orf72":  "ALS gene",    # Hexanucleotide repeat expansion; ubiquitous expression
    "TBK1":     "ALS gene",    # TANK-binding kinase 1; ALS/FTD causative; broadly expressed
    "OPTN":     "ALS gene",    # Optineurin; ALS causative; broadly expressed
    "VCP":      "ALS gene",    # Valosin-containing protein; ALS/IBM/Paget; ubiquitous
    "NEK1":     "ALS gene",    # Never in mitosis A-related kinase 1; ALS risk; broad
    "ATXN2":    "ALS gene",    # Ataxin 2; ALS modifier/risk; Takeuchi 2025: microglia+others
    "UNC13A":   "ALS gene",    # Munc13-1; ALS GWAS; TDP-43 target; enriched in neurons/MN
    "HNRNPA1":  "ALS gene",    # Heterogeneous nuclear RNP A1; ALS causative RBP; pan-neuronal
    "ANXA11":   "ALS gene",    # Annexin A11; ALS causative; widely expressed
    "KIF5A":    "ALS gene",    # Kinesin heavy chain 5A; ALS risk; motor neurons+other neurons
    "PFN1":     "ALS gene",    # Profilin 1; ALS causative; ubiquitous cytoskeletal protein
    "TUBA4A":   "ALS gene",    # Tubulin alpha 4A; ALS causative; broadly expressed
    "STMN2":    "Motor Neuron", # (already classified above; TDP-43 regulated MN marker)
    "SMCR8":    "ALS gene",    # Smith-Magenis chromosome region 8; C9orf72 complex; broad
    "GLT8D1":   "ALS gene",    # Glycosyltransferase 8 domain 1; ALS GWAS locus
    "PSEN1":    "ALS gene",    # Presenilin 1; AD/ALS-FTD; widely expressed
    "APP":      "ALS gene",    # Amyloid precursor protein; AD/ALS context; broad neuronal
    "LRRK2":    "ALS gene",    # Leucine-rich repeat kinase 2; PD/ALS GWAS; microglia+neurons
    "BAG3":     "ALS gene",    # BCL2-associated athanogene 3; ALS/CMT; widely expressed
    "CASP8":    "ALS gene",    # Caspase 8; ALS neuroinflammation; broadly expressed
    "TP53":     "ALS gene",    # Tumour suppressor; ALS modifier context; ubiquitous
    "RIPK3":    "ALS gene",    # Receptor-interacting kinase 3; necroptosis; ALS-relevant; broad
    "NCSTN":    "ALS gene",    # Nicastrin; gamma-secretase; ALS modifier; broad
    "PARP4":    "ALS gene",    # Poly-ADP-ribose polymerase 4; DNA repair; broad

    # -------------------------------------------------------------------------
    # BROAD / MULTIPLE (genuinely pan-cellular or bimodal markers)
    # -------------------------------------------------------------------------
    "APOE":     "Microglia",   # (final: microglia-dominant in CNS, already listed)
    "STAT5A":   "Broad/Multiple",   # STAT5A; haematopoietic + broad glial signalling
    "NOTCH1":   "Endothelial",      # (final assignment endothelial)
    "KIT":      "Broad/Multiple",   # c-KIT; mast cells / some progenitors / broad
    "NIPA1":    "Broad/Multiple",   # Non-imprinted Prader-Willi/Angelman 1; broad neuronal
    "RIT2":     "Broad/Multiple",   # RIT2 GTPase; dopaminergic + broad neuronal
    "GADD45G":  "Broad/Multiple",   # Growth arrest and DNA damage 45G; broad stress response
    "GEM":      "Broad/Multiple",   # GTP-binding protein GEM; broad / vascular smooth muscle
    "GOLM1":    "Broad/Multiple",   # Golgi membrane protein 1; Golgi; broad expression
    "FBP1":     "Broad/Multiple",   # Fructose-1,6-bisphosphatase; metabolic; liver>>CNS
    "SEPHS2":   "Broad/Multiple",   # Selenophosphate synthetase 2; ubiquitous selenoprotein
    "MID1":     "Broad/Multiple",   # Midline 1; broadly expressed ubiquitin ligase
    "PHLDB2":   "Broad/Multiple",   # Pleckstrin homology-like domain B2; broad expression
    "AGPAT3":   "Broad/Multiple",   # Acylglycerophospholipid transferase; broad lipid synthesis
    "GNB4":     "Broad/Multiple",   # G-protein beta 4; broad neuronal + immune
    "SCFD1":    "Broad/Multiple",   # Sec1 family domain 1; vesicle trafficking; broad
    "NUCKS1":   "Inhibitory Interneuron",  # (final: see inhibitory neuron block above)
    "STAT3":    "Astrocyte",        # (final: reactive astrocyte, see astrocyte block)
    "DDI2":     "Broad/Multiple",   # DNA damage-inducible protein 1; ubiquitin-proteasome; broad
    "FAR2":     "Broad/Multiple",   # Fatty acyl-CoA reductase 2; lipid metabolism; broad
    "EXOG":     "Broad/Multiple",   # Endo/exonuclease G; mitochondrial DNA repair; broad
    "EPB41L5":  "Broad/Multiple",   # Band 4.1-like protein 5; broad epithelial/neuronal
    "GRB2":     "Broad/Multiple",   # Growth factor receptor-bound 2; universal signal adaptor
    "PM20D1":   "Broad/Multiple",   # Peptidase M20 domain 1; broad metabolic
    "ADPGK":    "Broad/Multiple",   # ADP-dependent glucokinase; broad metabolic
    "CREG1":    "Broad/Multiple",   # Cellular repressor of E1A; broad secreted glycoprotein
    "SARDH":    "Broad/Multiple",   # Sarcosine dehydrogenase; mitochondrial; broad
    "RBM20":    "Broad/Multiple",   # RNA-binding motif 20; cardiac >> CNS; broad
    "EYA4":     "Broad/Multiple",   # Eyes absent 4; upper MN marker (cortex) / broad; spinal cord not restricted
    "THSD4":    "Endothelial",      # (final: endothelial, see above)
    "PON3":     "Broad/Multiple",   # Paraoxonase 3; liver/vascular; limited SC expression
    "DAAM2":    "Oligodendrocyte",  # (final: see OL block)
    "MICAL1":   "Broad/Multiple",   # Molecule interacting with CasL 1; broad cytoskeletal

    # -------------------------------------------------------------------------
    # UNKNOWN / lncRNA / NOVEL TRANSCRIPTS
    # Novel loci, antisense RNAs, or genes with insufficient CNS cell-type data
    # -------------------------------------------------------------------------
    "AC009242.1":   "Unknown/lncRNA",
    "AC010201.2":   "Unknown/lncRNA",
    "AC011287.1":   "Unknown/lncRNA",
    "AC013287.1":   "Unknown/lncRNA",
    "AC096667.1":   "Unknown/lncRNA",
    "AC099792.1":   "Unknown/lncRNA",
    "AC119673.2":   "Unknown/lncRNA",
    "AJ009632.2":   "Unknown/lncRNA",
    "AL136456.1":   "Unknown/lncRNA",
    "ID2-AS1":      "Unknown/lncRNA",   # ID2 antisense; no specific spinal cord cell-type data
    "LINC01322":    "Unknown/lncRNA",
    "LINC02099":    "Unknown/lncRNA",
    "LINC02232":    "Unknown/lncRNA",
    "MIR137HG":     "Unknown/lncRNA",   # miR-137 host gene; schizophrenia locus; neurons broadly
    "MIR497HG":     "Unknown/lncRNA",   # miR-497/195 cluster host gene; broad
    "NPSR1-AS1":    "Unknown/lncRNA",
    "NR2F2-AS1":    "VLMC",            # NR2F2 locus antisense — VLMC (see VLMC block)
    "OBI1-AS1":     "Unknown/lncRNA",
    "SCN1A-AS1":    "Unknown/lncRNA",   # SCN1A antisense; neuronal locus; no cell-type restriction
    "AQP4-AS1":     "Astrocyte",        # AQP4 antisense — astrocyte locus (see Astrocyte block)

    # -------------------------------------------------------------------------
    # ADDITIONAL GENES NOT GROUPED ABOVE — ALPHABETICALLY RESOLVED
    # -------------------------------------------------------------------------

    # Microglia / immune (additional)
    "ADGRG7":   "Microglia",          # Adhesion GPCR G7; microglia subtype enriched
    "BCL6":     "T cell",             # (already listed)
    "BTK":      "Microglia",          # Bruton's tyrosine kinase; microglia / B cells
    "CCL4":     "Microglia",          # CCL4/MIP-1beta; microglia activation chemokine
    "CD300A":   "Microglia",          # (already listed)
    "FERMT3":   "Microglia",          # Kindlin-3; haematopoietic integrin activation; microglia
    "GPSM3":    "Microglia",          # G-protein signalling modulator 3; myeloid cells
    "IL1A":     "Microglia",          # IL-1 alpha; microglia/macrophage inflammasome
    "IL1RAPL2": "Excitatory Interneuron", # IL-1R accessory protein-like 2; neuronal synapse
    "IL17RD":   "Broad/Multiple",     # IL-17 receptor D; broad; limited CNS data
    "SAMD5":    "Microglia",          # SAM domain 5; myeloid/microglia
    "SLC15A3":  "Microglia",          # Peptide/histidine transporter; microglia lysosome
    "SLC46A3":  "Microglia",          # SLC46A3; lysosomal transporter; microglia
    "SLC25A33": "Broad/Multiple",     # Mitochondrial pyrimidine transporter; ubiquitous
    "SLC25A48": "Broad/Multiple",     # Mitochondrial transporter; ubiquitous
    "SLC26A4":  "Broad/Multiple",     # Pendrin; inner ear >> CNS; limited SC specificity
    "SLC41A1":  "Broad/Multiple",     # Magnesium transporter; broad expression
    "SLC24A3":  "Excitatory Interneuron", # NCKX3; SC excitatory neuron enriched
    "STXBP2":   "Microglia",          # (already listed)
    "TGFB1":    "Monocyte/Macrophage", # (already listed)
    "TREM2":    "Microglia",          # (already listed)
    "TRIM38":   "Microglia",          # Tripartite motif 38; innate immune; microglia
    "UNC93B1":  "Microglia",          # Unc-93 homolog B1; TLR trafficking; microglia
    "VAV2":     "Microglia",          # VAV2 guanine nucleotide exchange factor; myeloid/microglia

    # Neuronal (additional — not fitting cleanly above)
    "ADGRV1":   "Excitatory Interneuron", # VLGR1; hair cell/neuron; SC sensory neurons
    "ALK":      "Motor Neuron",           # Anaplastic lymphoma kinase; motor neuron trophic
    "ATP10A":   "Excitatory Interneuron", # P4-ATPase; neuronal membrane lipid flip; SC neurons
    "B3GAT2":   "Excitatory Interneuron", # Beta-1,3-glucuronyltransferase; dorsal horn marker
    "BTBD11":   "Inhibitory Interneuron", # BTB domain 11; V1 spinal inhibitory interneurons (Yadav 2023)
    "CAPN3":    "Motor Neuron",           # Calpain 3; motor neuron / skeletal muscle (SC MNs)
    "CDH1":     "Inhibitory Interneuron", # E-cadherin; dorsal horn inhibitory laminar marker
    "CLSTN2":   "Excitatory Interneuron", # Calsyntenin 2; SC excitatory synapse protein
    "CNTNAP3B": "Excitatory Interneuron", # Contactin-associated protein-like 3B; SC neurons
    "CTNNA3":   "Excitatory Interneuron", # Alpha-catenin 3; SC neuronal junctions
    "CUX2":     "Excitatory Interneuron", # CUX2 TF; layer 2/3 excit. cortex, SC dorsal horn
    "EFHD1":    "Endothelial",            # (see endothelial above)
    "EPHA4":    "Motor Neuron",           # EphA4 receptor; motor neuron circuit formation
    "ERBB3":    "OPC",                    # ErbB3 / HER3; OPC/Schwann cell neuregulin receptor
    "ERICD":    "Excitatory Interneuron", # ERIC domain; SC neuronal
    "EYA4":     "Broad/Multiple",         # (see above; upper MN cortex context)
    "FILIP1":   "Excitatory Interneuron", # Filamin A-interacting protein 1; SC excit. neurons
    "FNIP2":    "Broad/Multiple",         # FNIP2; mTOR complex; broad haematopoietic + neuronal
    "FRMD4B":   "Excitatory Interneuron", # FERM domain B; SC dorsal horn neurons
    "FSTL4":    "Excitatory Interneuron", # Follistatin-like 4; SC dorsal interneurons
    "FSTL5":    "Excitatory Interneuron", # Follistatin-like 5; SC excitatory interneurons
    "GADD45G":  "Broad/Multiple",         # (see broad above)
    "GALNT14":  "Excitatory Interneuron", # GalNAc transferase 14; SC dorsal neurons
    "GAS2L3":   "Broad/Multiple",         # GAS2-like 3; cytoskeleton; broad
    "GMIP":     "Excitatory Interneuron", # GEM-interacting protein; SC neuronal
    "GSTM5":    "Broad/Multiple",         # Glutathione S-transferase mu 5; broad
    "HFE":      "Microglia",              # Hereditary haemochromatosis; iron regulation; microglia
    "HHATL":    "Oligodendrocyte",        # Hedgehog acyltransferase-like; SC OL enriched
    "IGFBP4":   "Broad/Multiple",         # IGF-binding protein 4; astrocyte + broad
    "IGSF1":    "Broad/Multiple",         # Immunoglobulin SF member 1; pituitary >> broad CNS
    "KANK1":    "Broad/Multiple",         # KN motif ankyrin 1; cytoskeletal; broad
    "KIF22":    "Broad/Multiple",         # Kinesin 22; mitotic spindle; broad proliferating
    "KIF26B":   "Excitatory Interneuron", # Kinesin 26B; SC interneuron development
    "KIF4A":    "Broad/Multiple",         # Kinesin 4A; mitotic; broad proliferating
    "KLK6":     "Oligodendrocyte",        # Kallikrein 6; SC oligodendrocyte serine protease
    "KNTC1":    "Broad/Multiple",         # Kinetochore-associated 1; mitotic; broad
    "LRP8":     "Excitatory Interneuron", # LRP8/ApoER2; excitatory neuron Reelin receptor
    "MEPE":     "Fibroblast",             # Matrix extracellular phosphoglycoprotein; meningeal FB
    "MFNG":     "OPC",                    # Manic fringe; Notch modifier; OPC/progenitors
    "MYO16":    "Excitatory Interneuron", # Myosin XVI; SC neuronal cytoskeleton
    "MYO5B":    "Excitatory Interneuron", # Myosin Vb; neuronal vesicle transport
    "NDST4":    "Excitatory Interneuron", # N-deacetylase/N-sulfotransferase 4; dorsal horn neurons
    "NES":      "OPC",                    # Nestin; NSC/OPC progenitor (also reactive astrocyte)
    "NPNT":     "Fibroblast",             # Nephronectin; ECM; meningeal fibroblast
    "NWD2":     "Excitatory Interneuron", # NACHT/WD40 domain 2; SC neuronal
    "OSBPL2":   "Broad/Multiple",         # Oxysterol-binding protein-like 2; broad lipid
    "OTOGL":    "Fibroblast",             # Otogelin-like; inner ear ECM; meningeal/vascular
    "PARP15":   "Microglia",              # (already listed)
    "PDE3B":    "Broad/Multiple",         # Phosphodiesterase 3B; broad metabolic / vascular
    "PLCE1":    "Excitatory Interneuron", # PLCepsilon1; SC excitatory signalling
    "PLCH1":    "Excitatory Interneuron", # PLCeta1; SC neuronal
    "PLD5":     "Excitatory Interneuron", # Phospholipase D5; SC neuronal
    "PTCHD4":   "Excitatory Interneuron", # Patched domain-containing 4; SC dorsal neurons
    "PTGS1":    "Microglia",              # COX-1; prostaglandin synthesis; microglia
    "PTPRK":    "Oligodendrocyte",        # PTPRK; OL differentiation phosphatase
    "RAB42":    "Broad/Multiple",         # RAB GTPase 42; vesicle trafficking; broad
    "RBFOX3":   "Motor Neuron",           # NeuN; pan-neuronal nuclear marker; strongest in MNs
    "RCSD1":    "Broad/Multiple",         # CAPZIP; actin cytoskeleton; broad
    "RHPN1":    "Broad/Multiple",         # Rhophilin 1; cytoskeletal; broad
    "ROS1":     "Broad/Multiple",         # ROS1 receptor tyrosine kinase; limited CNS data
    "RXFP1":    "Excitatory Interneuron", # Relaxin receptor 1; SC interneurons
    "S100Z":    "Broad/Multiple",         # S100 protein Z; limited specific CNS data
    "SCUBE1":   "Endothelial",            # Signal peptide CUB EGF 1; endothelial
    "SCUBE3":   "Endothelial",            # Signal peptide CUB EGF 3; endothelial
    "SEMA3E":   "Excitatory Interneuron", # Semaphorin 3E; SC axon guidance; excitatory circuits
    "SEMA5A":   "Excitatory Interneuron", # Semaphorin 5A; SC interneuron axonal guidance
    "SFTA3":    "Unknown/lncRNA",         # Surfactant protein A-related; lung >> CNS; unknown SC
    "SMCR8":    "ALS gene",               # (already listed)
    "SNTB2":    "Excitatory Interneuron", # Syntrophin beta 2; SC neuronal membrane scaffold
    "SOX9":     "Astrocyte",              # (already listed)
    "SPHKAP":   "Inhibitory Interneuron", # SKIP; PKA anchoring; SC inhibitory neurons
    "STAC3":    "Broad/Multiple",         # SH3 and cysteine-rich domain 3; skeletal muscle/NMJ; not SC MN intrinsic
    "STX3":     "Broad/Multiple",         # Syntaxin 3; vesicle fusion; broad neuronal
    "TENM1":    "Excitatory Interneuron", # Teneurin 1; SC excitatory circuit wiring
    "TF":       "Oligodendrocyte",        # Transferrin; SC oligodendrocyte iron transport
    "TOB2":     "Broad/Multiple",         # Transducer of ERBB2; broadly expressed
    "TRIM38":   "Microglia",              # (already listed)
    "TTN":      "Broad/Multiple",         # Titin; cardiac/skeletal muscle >>> CNS; SC panel likely for contamination QC
    "VIP":      "Inhibitory Interneuron", # (already listed)
    "ZBBX":     "Ependymal",              # Zinc finger SWIM-type; SC ependymal enriched (motile cilia)
    "ZDHHC23":  "Excitatory Interneuron", # DHHC23 palmitoyl transferase; SC neuronal
    "ZNF804A":  "Excitatory Interneuron", # Zinc finger 804A; schizophrenia GWAS; SC excit. neurons
    "ZNF804B":  "Excitatory Interneuron", # Zinc finger 804B; related locus; SC neurons
    "WWC3":     "Broad/Multiple",         # WWC3; Hippo pathway; broad CNS
    "WDR49":    "Astrocyte",              # WD repeat domain 49; ALS astrocyte-enriched (Milo/Rodrigo dataset)
    "SAMD9":    "Monocyte/Macrophage",    # (already listed)
    "COBLL1":   "Endothelial",            # (already listed)
    "ANKRD18A": "Unknown/lncRNA",         # Ankyrin repeat domain 18A; no spinal cell-type specificity
    "C22orf39":  "Unknown/lncRNA",        # Chromosome 22 open reading frame; limited data
    "ARL11":    "Microglia",              # ADP-ribosylation factor-like 11; myeloid/microglia
    "ARRB2":    "Microglia",              # Beta-arrestin 2; microglia GPCR signalling
    "ASCL2":    "OPC",                    # Achaete-scute TF 2; OPC/neural progenitor
    "ATP2C2":   "Inhibitory Interneuron", # SPCA2 Ca2+ pump; dorsal horn/SC neurons
    "BCAT1":    "Astrocyte",              # Branched-chain aminotransferase 1; astrocyte BCAA metabolism
    "CAV1":     "Endothelial",            # Caveolin 1; endothelial caveolae / pericyte
    "CCDC146":  "Ependymal",              # Coiled-coil domain 146; cilia/ependymal
    "CDCP1":    "Endothelial",            # CUB domain-containing 1; endothelial cell adhesion
    "CXCL14":   "Astrocyte",              # CXCL14; SC astrocyte-secreted chemokine (Blum 2021)
    "EFHD1":    "Endothelial",            # (already above)
    "FBLN1":    "Fibroblast",             # (already listed)
    "GNB4":     "Broad/Multiple",         # (already listed)
    "GPSM3":    "Microglia",              # (already listed)
    "KIAA1217": "Excitatory Interneuron", # KIAA1217; SC excitatory neuron enriched
    "KLK6":     "Oligodendrocyte",        # (already listed)
    "LRRK2":    "ALS gene",               # (already listed)
    "LSP1":     "T cell",                 # (already listed)
    "LYVE1":    "Microglia",              # (already listed; perivascular macrophage)
    "MEGF11":   "OPC",                    # (already listed)
    "MEIS2":    "Inhibitory Interneuron", # (already listed)
    "MNDA":     "Microglia",              # (already listed)
    "MPP1":     "Schwann",                # (already listed)
    "MX2":      "Monocyte/Macrophage",    # (already listed)
    "NCF2":     "Microglia",              # (already listed)
    "NELL2":    "Excitatory Interneuron", # (already listed)
    "NOTCH1":   "Endothelial",            # (already listed)
    "RUBCNL":   "Microglia",              # Rubicon-like; autophagy; microglia enriched
    "SNTB2":    "Excitatory Interneuron", # (already listed)
    "STAC3":    "Broad/Multiple",         # (already listed)
    "TOM1L1":   "Broad/Multiple",         # Target of Myb 1-like 1; endosomal; broad
    "TRHDE":    "Excitatory Interneuron", # Thyrotropin-releasing hormone-degrading enzyme; SC neurons
    "C3AR1":    "Microglia",              # (already listed)
    "CFAP410":  "Ependymal",              # (already listed)
    "DOCK2":    "Microglia",              # (already listed)
    "EPSTI1":   "Microglia",              # (already listed)
    "GMFG":     "Microglia",              # (already listed)
    "IFITM3":   "Microglia",              # Interferon-induced TM protein 3; microglia/macrophage
    "MFNG":     "OPC",                    # (already listed)
    "NPNT":     "Fibroblast",             # (already listed)
    "PARP15":   "Microglia",              # (already listed) 
}

# "Raw"

In [ ]:
# Load the sample. This h5ad already carries the Novae domain labels and
# the spatial coordinates. A raw copy of X is stashed in layers['counts']
# so normalisation can be redone from scratch in pass 2.

adata = sc.read("/oak/stanford/scg/lab_mpsnyder/johnck/Projects/RK/Spatial/Novae_persample_INDEPENDENT_FINAL/per_sample/SD01923_BI__niches_independent.h5ad")

if "counts" not in adata.layers:
    adata.layers["counts"] = adata.X.copy()

adata

### Correction 2026-08-17 — X is log-normalised, the counts are in a layer

`sc.read` returns an object whose `X` has **already** been normalised and
log1p-transformed by Novae (float32, max ~2.6). The genuine integer transcript
counts are in `layers['counts']`; their row sums equal `obs['transcript_counts']`.

The cell above only stashes `X` into that layer when it is *absent*, so the layer
is intact — but every QC cell below reads `X`. `sc.pp.calculate_qc_metrics`
overwrites `obs['total_counts']` with log-normalised row sums, and the Salas
`>= 10 transcripts` filter then thresholds that column. Applied to log row sums
the filter kept **8.4%** of the cohort (median sample 2.2%) and reduced
SD01915_BG, SD01922_BI and SD01923_BI to **zero cells**, which crashes the
spatial-neighbours step.

The tell was in the notebook's own output: a printed median of **8.2 transcripts**
per cell alongside a median of **11 detected genes**. Eleven genes cannot be
detected from 8.2 transcripts.

The next cell restores the notebook's own assumption before anything reads `X`.


In [ ]:
# ── SOURCE FIX 2026-08-17 — restore raw counts before any QC ─────────────
# X arrives normalised + log1p'd; layers['counts'] holds the integer counts.
# Restore them here so calculate_qc_metrics writes true totals and the Salas
# >= 10 transcript cut thresholds transcripts rather than log row sums.
import os as _os
import numpy as _np

assert 'counts' in adata.layers, "layers['counts'] absent -- cannot restore raw counts"
_pre  = float(_np.asarray(adata.X.sum(1)).ravel().mean())
adata.X = adata.layers['counts'].copy()
_post = _np.asarray(adata.X.sum(1)).ravel()
assert _np.allclose(_post, adata.obs['transcript_counts'].values, atol=1e-3), \
    "restored X row sums != obs['transcript_counts'] -- do not trust this run"
print(f"X row-sum mean {_pre:.2f} (log-normalised)  ->  {_post.mean():.2f} (raw counts)")
print(f"median transcripts/cell {adata.obs['transcript_counts'].median():.0f}; "
      f"{(adata.obs['transcript_counts'] < 10).mean()*100:.1f}% below the >= 10 cut")

# One path per object. Pass 1 wrote one filename and pass 2 read another (the
# '_fisrt_pass_filtering' typo), so pass 2 could resume from a stale object
# without erroring. Env-overridable so a staging run cannot clobber canonical
# outputs.
SAMPLE_ID  = "SD01923_BI"
SAMPLE_TAG = "SD01923BI"
PASS1_H5AD = _os.environ.get(
    "DUALPASS_PASS1_H5AD",
    "/oak/stanford/scg/lab_mpsnyder/johnck/Projects/RK/Spatial/Novae_IND_QC/ALS_SCXenium_SD01923BI_postQC.h5ad")
PASS2_H5AD = _os.environ.get(
    "DUALPASS_PASS2_H5AD",
    "/oak/stanford/scg/lab_mpsnyder/johnck/Projects/RK/Spatial/Ranger_procd/ALS_SCXenium_SD01923BI_pass2.h5ad")
print("pass 1 ->", PASS1_H5AD)
print("pass 2 ->", PASS2_H5AD)

try:
    __TRACE__
except NameError:
    __TRACE__ = {}
__TRACE__.update({
    "sample": SAMPLE_ID,
    "fix_counts_applied": True,
    "true_median_transcripts": float(adata.obs['transcript_counts'].median()),
    "true_mean_transcripts": float(adata.obs['transcript_counts'].mean()),
    "true_pct_lt10": float((adata.obs['transcript_counts'] < 10).mean() * 100),
    "pass1_h5ad": PASS1_H5AD,
    "pass2_h5ad": PASS2_H5AD,
})


### Relabel v6 2026-08-18 - motor neurons from `neuron_classification_tdp_v6`

Cell calls come from `Ranger_procd_mw_final/annotation/neuron_classification_tdp_v6.parquet`,
read through the snapshot pinned at `neuron_classification_tdp_v6__snapshot_20260818_1649.parquet`
(md5 `e4f7ffc0fb8ebe794c4f0c7b1bce4095`, asserted below). **This replaces the relabel run
earlier today, which read the v2 file by mistake. Every number from that run is superseded.**

The v4 rule, exact (0 false positives, 0 false negatives) on each of the 22 sample keys
individually:

    is_MN = is_neuron_v3 & ~is_GAD & in_VH & (cell_area_um2 >= 250)

Four things about v4 that the v2 relabel got wrong or could not know:

- **`in_VH` is no longer a subset of `in_GM`.** 6,902 violations across 12 of our 20 sections,
  all of them in `robin_pathologist` samples. The v2 notebook asserted the subset relation;
  that assert is replaced here by a counted report written to `obs['in_VH_not_GM']` and to the
  trace. 35 of the 738 called motor neurons sit outside grey matter, which is anatomically
  impossible, so this is a mask-registration defect and not biology.
- **v4 ships 16 exactly-duplicated `(sample, cell_index)` pairs.** Deduplicated, the file is
  bit-exact the Jul-22 universe. The join drops them after asserting byte-identity. Over our
  20 sections the true counts are **599 motor neurons, not 601**, and **6,035 neurons, not
  6,037** -- all of the difference is in SD04219_BI.
- **`is_GAD` and `is_Interneuron` are no longer neuron subsets:** 455 of 474 GAD+ and 1,065 of
  1,237 Interneuron cells are non-neurons. `is_gabaergic` is therefore renamed
  `is_gabaergic_neuron`.
- **v4 has no nulls in any label column.** `has_vh_data = in_VH.notna()` would be True for all
  20 sections, which silently converts "no ventral-horn mask" into "zero motor neurons". It is
  redefined here as a measured count. The three columns that *do* carry nulls are
  `pathologist_class`, `cord_level` (100% null on 4 whole samples) and `tdp_proba` (281,859
  ALS cells, 0 Control).

`SD01015_BG` and `SD01616_BI` have `in_GM` True for **0** cells and `in_VH` True for **0**
cells. That is a missing mask, not a measurement: 71 and 138 cells respectively satisfy
everything except `in_VH`. Because `.sum()` skips `<NA>`, the only reduction-proof contract is
`is_MN = <NA>` for *every* cell in such a section, which is what this cell writes -- so a mean
is `NaN` and a bar chart has a gap rather than a zero. `is_MN_eligible` (no `in_VH` term) is
written on every section so an MN rate always has an honest denominator.

The join is gated on **spatial coordinates** (max |delta| < 1e-3 um; measured 1.8e-12 um), not
on row counts. A count picks the wrong member of the SD02022 pair. `is_TDP` and `tdp_proba` are
carried but confounded with disease by construction and must stay out of any contrast, and
`pathologist_class` scores are reported as raw counts only -- the 183 labels are very likely
in-sample and are 129-fold enriched for neurons relative to the file.


In [ ]:

# ── VH RELABEL V6 2026-08-18 ──────────────────────────────────────────────
# v4 cell calls. is_MN = is_neuron_v3 & ~is_GAD & in_VH & area >= 250um2.
#
# v4 has no nulls in the label columns, so nothing here NEEDS NA-safety today.
# The .astype("boolean").fillna(False) discipline is kept anyway: pd.NA is
# truthy, `== True` KEEPS <NA>, and this cell itself deliberately introduces
# <NA> on a section whose ventral-horn mask is missing.
import hashlib as _hashlib
import numpy as _np
import pandas as _pd

VH_PARQUET = ("/oak/stanford/scg/lab_mpsnyder/johnck/Projects/RK/Spatial/Novae_IND_QC/runs_srcfixed/annotation_snapshot/neuron_classification_tdp_v6__snapshot_20260818_1649.parquet")
VH_PARQUET_MD5 = "d32b298e9662e023cc1794367680a0ce"
VH_PARQUET_SAMPLE = "SD01923_BI"      # verified by coordinate agreement, not by name
AREA_MIN = 250.0              # um^2, the v4 motor-neuron size gate. The file
                                    # only constrains it to (249.0152, 251.5203]
EXPECTED_DUP_ROWS = 0       # exact-duplicate cell_index rows in this sample
EXPECTED_NATIVE_MN = 10   # our markers+size call on the PRISTINE input
GAD_THRESHOLD = 8                   # only for the independent cross-check below

# md5 the snapshot. An array job takes minutes per sample; if somebody re-pins
# the snapshot mid-flight, the samples before and after would carry different
# labels in one cohort object and nothing would say so.
_h = _hashlib.md5()
with open(VH_PARQUET, "rb") as _fh:
    for _chunk in iter(lambda: _fh.read(1 << 22), b""):
        _h.update(_chunk)
assert _h.hexdigest() == VH_PARQUET_MD5, (
    f"{VH_PARQUET} md5 is {_h.hexdigest()}, expected {VH_PARQUET_MD5} -- the "
    f"pinned snapshot was rewritten; do not mix labels across one cohort")

_cols = ["cell_index", "x_centroid", "y_centroid",
         "is_neuron_v3", "neuron_prob_v3", "in_GM", "in_VH",
         "is_GAD", "is_Interneuron", "cell_area_um2", "mn_signature_prob",
         "is_MN", "cell_type", "is_TDP", "tdp_proba", "STMN2", "CE_STMN2",
         "pathologist_class", "vh_source",
         # vh_annotated is new in v5 and is READ by the VH-presence check below.
         # It has to be in this list: the slice is taken as _vh.loc[..., _cols],
         # so a column read but not listed here raises KeyError at runtime while
         # compiling and rendering perfectly. That cost a 20-task array.
         "vh_annotated",
         "cord_level", "disease_group", "patient_id"]
_vh = _pd.read_parquet(VH_PARQUET)
_missing = [c for c in _cols if c not in _vh.columns]
assert not _missing, (f"{VH_PARQUET} is missing {_missing}; the schema changed "
                      f"again -- it now has {list(_vh.columns)}")
_vh = _vh.loc[_vh["sample"] == VH_PARQUET_SAMPLE, _cols]
assert len(_vh), f"no rows for {VH_PARQUET_SAMPLE} in {VH_PARQUET}"

# ── v4 ships 16 exactly-duplicated (sample, cell_index) pairs ───
# reindex() raises ValueError on a duplicate index, so this kills 4 of the 20
# notebooks if left alone. Assert byte-identity BEFORE dropping: a conflicting
# pair would mean two different calls for one cell and keep="first" would pick
# one at random.
_dup_key = _vh.duplicated(subset=["cell_index"], keep=False)
_dup_all = _vh.duplicated(keep=False)
assert bool((_dup_key <= _dup_all).all()), (
    f"{VH_PARQUET_SAMPLE}: duplicated cell_index rows that DIFFER in some column "
    f"-- two different calls for one cell, refusing to pick one")
_n_dup = int(_vh.duplicated(subset=["cell_index"], keep="first").sum())
assert _n_dup == EXPECTED_DUP_ROWS, (
    f"{VH_PARQUET_SAMPLE}: {_n_dup} duplicate cell_index rows, expected "
    f"{EXPECTED_DUP_ROWS}. The duplication pattern changed -- update "
    f"EXPECTED_DUPS in relabel_v4real_at_source.py and re-check the counts")
_vh = _vh.drop_duplicates(subset=["cell_index"], keep="first")

# ── nulls: which columns are ALLOWED to carry them ──────────
# v4 had no nulls anywhere, so this asserted a blanket no-null contract. v5 and v6
# deliberately reintroduced them, and the blanket assert then killed exactly the
# two sections the nulls exist to protect (SD01015_BG in_VH 51,912 / is_MN 201;
# SD01616_BI in_VH 80,312 / is_MN 324). Allowed, with a reason each:
#   cord_level        100% null on 4 whole samples (metadata gap, not measured)
#   tdp_proba         NaN for every control by construction, plus 24.4% of ALS
#   pathologist_class 183 hand labels in the whole file, null elsewhere
#   in_VH, is_MN      NULL BY DESIGN on a section with no ventral-horn mask.
#                     This is the whole point of v5's vh_annotated: "not
#                     annotated" must not be encodable as False.
# Everything else must still be null-free, so a genuine schema slip is caught.
_nullable_ok = ("cord_level", "tdp_proba", "pathologist_class", "in_VH", "is_MN")
_nulls = {_c: int(_vh[_c].isna().sum()) for _c in _cols}
_bad_nulls = {_c: _n for _c, _n in _nulls.items()
              if _n and _c not in _nullable_ok}
assert not _bad_nulls, (f"{VH_PARQUET_SAMPLE}: nulls in columns that must not have "
                        f"them: {_bad_nulls}. in_VH/is_MN nulls are expected on a "
                        f"mask-less section; these are not those")
# and the nulls that ARE allowed still have to be self-consistent: in_VH is either
# fully null (no mask) or fully present, never partial, and is_MN nulls are a
# subset of the in_VH nulls
_n_invh_null = int(_vh["in_VH"].isna().sum())
assert _n_invh_null in (0, len(_vh)), (
    f"{VH_PARQUET_SAMPLE}: in_VH is null on {_n_invh_null} of {len(_vh)} rows. "
    f"A partial ventral-horn mask has no defined meaning here")
assert not bool((_vh["is_MN"].isna() & _vh["in_VH"].notna()).any()), (
    f"{VH_PARQUET_SAMPLE}: is_MN is null where in_VH is not; is_MN nulls must be "
    f"a subset of the mask-less rows")
print(f"nulls allowed and present: "
      f"{ {_c: _n for _c, _n in _nulls.items() if _n} }")

# obs_names are "<sample>:<cells.parquet cell_id>", and cell_index is cell_id - 1.
# The -1 was tested against both neighbours: +0 and -2 each give a coordinate
# agreement fraction of exactly 0.000 on all 20 samples.
_ci = _np.array([int(str(o).split(":")[1]) - 1 for o in adata.obs_names])
_m = _vh.set_index("cell_index").reindex(_ci)
_joined = int(_m["is_neuron_v3"].notna().sum())
print(f"joined {_joined:,} / {adata.n_obs:,} cells to {VH_PARQUET_SAMPLE} "
      f"({_n_dup} duplicate parquet rows dropped)")
assert _joined == adata.n_obs, (f"only {_joined} of {adata.n_obs} cells joined; "
                                f"cell_index is not aligned")

# ── THE JOIN GATE: coordinates, not counts ──────────────────
# A row count cannot prove a join, and here it actively misleads: our SD02022_BI
# raw 76,564 is closer to decoy SD02022 79,637 than to the true SD020_22_BI
# 80,618. The true sample agrees to 1.8e-12 um and the closest wrong key in the
# file is 2,670 um away in median, so this proves the join AND the sample-name
# mapping in one step. It is also the only thing that catches the wrong BUILD:
# the AUG5 objects index-join at 88-99% and place 392 of 599 motor neurons on
# the wrong cell with no index miss to raise on.
_xy = _np.asarray(adata.obsm["spatial"], dtype=float)
_dx = _np.abs(_xy[:, 0] - _m["x_centroid"].to_numpy(dtype=float))
_dy = _np.abs(_xy[:, 1] - _m["y_centroid"].to_numpy(dtype=float))
_maxd = float(_np.nanmax(_np.maximum(_dx, _dy)))
print(f"coordinate content gate: max |delta| = {_maxd:.3e} um")
assert _maxd < 1e-3, (f"spatial coordinates disagree by {_maxd:.3f}um -- this is "
                      f"either the WRONG parquet sample or the WRONG BUILD for "
                      f"this object. Do not relax this threshold")

# ── is the ventral-horn mask present at all? ────────────────
# v6 answers this three ways and we require all three to agree, because each one
# alone has failed at some point in this file's history:
#   (a) `vh_annotated`, Marcel's explicit flag, new in v5. Authoritative.
#   (b) in_VH.isna(), which is now meaningful (it was uniformly False in v4).
#   (c) DERIVED from the section's own geometry, which is what we used on v4 and
#       is the only one that keeps working if a future version drops the flag.
# NEVER vh_source: it reads 'auto_density_fixed' on both mask-less sections.
# NA-safe reads throughout: v6's in_VH is nullable, and .to_numpy(dtype=bool) on
# a nullable column containing pd.NA raises, so the bare v4 form would die here.
_in = _m["in_VH"].astype("boolean").fillna(False).to_numpy(dtype=bool)
_gm = _m["in_GM"].astype("boolean").fillna(False).to_numpy(dtype=bool)
_in_na = _m["in_VH"].isna().to_numpy()
_n_in_vh = int(_in.sum())
_n_in_gm = int(_gm.sum())

_flag_annotated = bool(_pd.Series(_m["vh_annotated"]).astype("boolean")
                       .fillna(False).all())
_na_says_present = not bool(_in_na.all())
_geom_says_present = bool(_n_in_vh > 0 or _n_in_gm > 0)
assert _flag_annotated == _na_says_present == _geom_says_present, (
    f"{VH_PARQUET_SAMPLE}: the three VH-presence signals disagree -- "
    f"vh_annotated={_flag_annotated}, in_VH-not-all-NA={_na_says_present}, "
    f"geometry={_geom_says_present}. One of them is now wrong and the MN call "
    f"for this section cannot be trusted until that is resolved by hand")
VH_MASK_PRESENT = _flag_annotated
VH_STATUS = "present" if VH_MASK_PRESENT else "absent_no_gm_no_vh"
# v6 also carries partial NA: is_MN is <NA> only on the confirmed neurons of a
# mask-less section, while its glia stay False. Recorded so the contract below is
# auditable against the delivery rather than reconstructed from it.
_n_isMN_na_delivered = int(_m["is_MN"].isna().sum())


def _nullable_vh(vals):
    """bool array -> pandas 'boolean'; all-<NA> when this section has no mask.

    All-<NA> and not just-the-candidates-<NA>: pandas .sum() SKIPS <NA>, so a
    column of False plus a handful of <NA> still sums to 0 and a reader sees a
    measurement. With every cell <NA>, .mean() is NaN and .sum(min_count=1) is
    NaN, which is what an undetermined section should look like.
    """
    if VH_MASK_PRESENT:
        return _pd.array(_np.asarray(vals, dtype=bool), dtype="boolean")
    return _pd.array([None] * len(vals), dtype="boolean")


# ── carry the columns ───────────────────────────────────────
# is_neuron_general must be captured BEFORE is_MN is overwritten. On a pristine
# notebook obs['is_MN'] is our own markers+size call; if this notebook had been
# patched on top of an earlier relabel it would be that relabel's MN call
# instead, which is what the assert below catches.
# read through astype("boolean").fillna(False), not np.asarray().astype(bool):
# the pristine column is plain bool with 0 nulls today (measured on all 20), but
# np.asarray on a nullable column turns pd.NA into True, i.e. into a motor neuron
adata.obs["is_neuron_general"] = (
    _pd.Series(adata.obs["is_MN"]).astype("boolean").fillna(False)
    .to_numpy(dtype=bool))
_n_gen = int(adata.obs["is_neuron_general"].sum())
assert _n_gen == EXPECTED_NATIVE_MN, (
    f"is_neuron_general is {_n_gen}, expected {EXPECTED_NATIVE_MN} from the "
    f"pristine markers+size call. This notebook was NOT restored from "
    f".bak_prev4relabel -- never patch on top of a patch")

adata.obs["is_neuron_prev"] = (
    _pd.Series(adata.obs["is_neuron"]).astype("boolean").fillna(False)
    .to_numpy(dtype=bool) if "is_neuron" in adata.obs.columns
    else _np.zeros(adata.n_obs, dtype=bool))
_n_prev = int(adata.obs["is_neuron_prev"].sum())

# DROP the stale is_neuron / neuron_prob. They already exist on the input object
# (10,085 over the 20 vs v4's 6,035), and writing v4 under the new names would
# leave every consumer keyed on the string 'is_neuron' reading the old column
# with no error -- measured as a 110% overstatement post-filter.
_dropped = [_c for _c in ("is_neuron", "neuron_prob") if _c in adata.obs.columns]
adata.obs = adata.obs.drop(columns=_dropped)

for _c in ("is_neuron_v3", "is_GAD", "is_Interneuron", "is_TDP"):
    adata.obs[_c] = _m[_c].to_numpy(dtype=bool)
# v6's is_MN is nullable (525 cohort-wide), so this must go through
# astype("boolean"), not .to_numpy(dtype=bool), which raises on pd.NA.
adata.obs["is_MN_v6_asdelivered"] = _pd.array(_m["is_MN"].to_numpy(), dtype="boolean")
for _c in ("cell_area_um2", "mn_signature_prob", "tdp_proba", "neuron_prob_v3"):
    adata.obs[_c] = _m[_c].to_numpy(dtype="float32")
for _c in ("STMN2", "CE_STMN2"):
    # renamed: an obs column called STMN2 would shadow the panel gene of the
    # same name in every scanpy colour lookup
    adata.obs[f"{_c}_mw"] = _m[_c].to_numpy(dtype="int32")

# in_GM / in_VH go <NA> on a mask-less section. in_VH's NAs now come FROM the
# file (v6 ships them); _nullable_vh reproduces the same thing and is kept so the
# behaviour is identical whether or not a future version supplies them.
adata.obs["in_GM"] = _nullable_vh(_gm)
adata.obs["in_VH"] = _nullable_vh(_in)

# has_vh_data was in_VH.notna(), which under v4 is True for 20/20 and destroys
# the sentinel. Redefined as the measured flag. Kept under the old name because
# promote_pass2.py and concat_pass2_v4.py require the column.
adata.obs["has_vh_data"] = _np.full(adata.n_obs, VH_MASK_PRESENT, dtype=bool)
adata.obs["vh_mask_empty"] = _np.full(adata.n_obs, not VH_MASK_PRESENT, dtype=bool)
adata.obs["vh_source"] = _pd.Categorical(
    _m["vh_source"].fillna("unassigned").astype(str).to_numpy())

# v4's cell_type carries no information (cell_type == 'motor_neuron' IS is_MN and
# 'other_neuron' IS is_neuron_v3 & ~is_MN, both exact on all 22 keys) and its
# 'glia' bucket is every non-neuron, endothelium and meninges included. Carried
# under _mw because this notebook assigns its OWN obs['cell_type'] from
# marker-gene argmax further down, and the astrocyte notebook selects
# cell_type == 'Astrocytes'.
adata.obs["cell_type_mw"] = _pd.Categorical(_m["cell_type"].astype(str).to_numpy())
# "unlabelled" rather than <NA> so a reader walking codes/categories directly
# cannot hit a -1 code
adata.obs["pathologist_class_mw"] = _pd.Categorical(
    _m["pathologist_class"].fillna("unlabelled").astype(str).to_numpy())
adata.obs["tdp_evaluated"] = _m["tdp_proba"].notna().to_numpy(dtype=bool)

_neu = adata.obs["is_neuron_v3"].to_numpy(dtype=bool)
_gad = adata.obs["is_GAD"].to_numpy(dtype=bool)
_int = adata.obs["is_Interneuron"].to_numpy(dtype=bool)
_mn_raw = (adata.obs["is_MN_v6_asdelivered"].astype("boolean")
           .fillna(False).to_numpy(dtype=bool))
_mn_na = adata.obs["is_MN_v6_asdelivered"].isna().to_numpy()
_area = adata.obs["cell_area_um2"].to_numpy()
_big = (_area >= AREA_MIN)

# ── the in_GM check, REPORTED not asserted ──────────────────
# v2 asserted `not (in_VH & ~in_GM).any()`. That is FALSE in v4 -- 6,902
# violations over our 20, in 12 of them, every one in a robin_pathologist
# sample and none in an auto_density_fixed one. Asserting it kills 12 of 20
# notebooks; with allow_errors=True it kills them QUIETLY, half-writing the
# contract. So it is counted, flagged per cell, and traced. in_GM is NOT applied
# to the MN call: the verified recipe does not use it.
adata.obs["in_VH_not_GM"] = (_in & ~_gm)
_n_vh_not_gm = int((_in & ~_gm).sum())
_n_mn_outside_gm = int((_mn_raw & ~_gm).sum())
if _n_vh_not_gm:
    print(f"in_VH outside in_GM: {_n_vh_not_gm:,} cells "
          f"({100.0 * _n_vh_not_gm / max(_n_in_vh, 1):.1f}% of in_VH); "
          f"{_n_mn_outside_gm} of them are called MN. Hand-drawn "
          f"robin_pathologist VH polygons extend past the auto GM mask -- a "
          f"motor neuron outside grey matter is anatomically impossible, so "
          f"treat this as mask registration, not biology")

# ── derived masks ───────────────────────────────────────────
# is_MN_eligible has NO in_VH term and is written on every section, so an MN
# rate has a denominator even where the numerator is undetermined.
adata.obs["is_MN_eligible"] = _neu & ~_gad & _big
adata.obs["is_MN_invh"] = _nullable_vh(_neu & _in)
adata.obs["is_GADvh"] = _nullable_vh(_gad & _in)
# renamed from is_gabaergic: in v4 is_GAD is NOT neuron-restricted, so this
# restriction discards most GAD+ cells and the name has to say so
adata.obs["is_gabaergic_neuron"] = _neu & _gad

# ── THE RECIPE GATE, v6: UNGATED ────────────────────────────
# v4/v5 baked an area >= 250 term into is_MN. v6 removes it, deliberately, so the
# threshold becomes our explicit downstream choice. The gate therefore asserts the
# UNGATED rule and would fire if the area term ever came back.
# Tested against the AS-DELIVERED column, on the non-NA rows only: a mask-less
# section has is_MN <NA> on its confirmed neurons, and pd.NA is truthy, so
# comparing those rows would silently pass whatever we asserted.
# No in_GM term: the verified recipe does not use it (and in v6 in_VH is not even
# a subset of in_GM -- see the counted report above).
_expect = _neu & ~_gad & _in
assert (_mn_raw[~_mn_na] == _expect[~_mn_na]).all(), (
    f"is_MN != is_neuron_v3 & ~is_GAD & in_VH on "
    f"{int((_mn_raw[~_mn_na] != _expect[~_mn_na]).sum())} of "
    f"{int((~_mn_na).sum())} determinate cells -- the v6 recipe changed. If the "
    f"area term has come back, this is a v5-style file and AREA_MIN belongs in "
    f"the rule again. Re-derive before trusting any motor-neuron count")
# and the NA pattern is itself a claim worth checking: <NA> should appear only on
# a mask-less section, and only on cells that are neurons
assert not bool(_mn_na.any()) or not VH_MASK_PRESENT, (
    f"is_MN has {int(_mn_na.sum())} <NA> cells on a section whose VH mask is "
    f"PRESENT -- that combination has no meaning")
assert not bool((_mn_na & ~_neu).any()), (
    f"is_MN is <NA> on {int((_mn_na & ~_neu).sum())} non-neurons; v6 leaves glia "
    f"in a mask-less section as False, so this is a different file than expected")

# ── the area threshold, now OURS to choose and to name ──────
# Marcel's v6 is_MN is the candidate pool. We keep an `is_MN` that means what it
# has always meant (area >= 250) so no downstream consumer silently changes
# meaning, and carry the alternatives beside it so a threshold question can be
# answered without regenerating anything.
#   250  Marcel's v4/v5 gate. `v6 is_MN & area >= 250` reproduces v5 exactly.
#   211  the 2-component mixture crossing on log10 area over the candidate pool
#        (bootstrap [188, 230]); also the Youden optimum against the pathologist
#        labels is a 204-220 plateau. Neither anchors to published human values;
#        see MN_AREA_THRESHOLD_RATIONALE.md.
# NOTE cell_area_um2 correlates with transcript count at rho ~0.82 inside this
# pool, so every area threshold is partly a depth filter. And 47 candidates
# cohort-wide have area exactly 0.0, which is a segmentation failure and not a
# small neuron, hence area_valid below rather than folding it into the cut.
adata.obs["is_MN_candidate"] = _nullable_vh(_expect)
adata.obs["area_valid"] = (_area > 0)
adata.obs["is_MN_a250"] = _nullable_vh(_expect & (_area >= 250.0))
adata.obs["is_MN_a211"] = _nullable_vh(_expect & (_area >= 211.0))
adata.obs["is_MN"] = _nullable_vh(_expect & _big)      # _big is AREA_MIN = 250
assert AREA_MIN == 250.0, (
    "obs['is_MN'] is documented everywhere as the area-250 call; changing "
    "AREA_MIN silently redefines it. Change is_MN_a<N> instead")

# independent cross-check of their GAD call against our own transcript counts.
# The candidate set does NOT assume is_GAD is neuron-restricted, because in v4
# it is not.
_g = []
for _gene in ("GAD1", "GAD2"):
    _col = adata.layers["counts"][:, list(adata.var_names).index(_gene)]
    _g.append(_np.asarray(_col.todense()).ravel() if hasattr(_col, "todense")
              else _np.asarray(_col).ravel())
_ours = (_g[0] >= GAD_THRESHOLD) | (_g[1] >= GAD_THRESHOLD)
_cand = _neu & _in
print(f"GAD among in-VH neurons: parquet {int((_gad & _cand).sum())}, "
      f"our GAD1/GAD2 >= {GAD_THRESHOLD} rule {int((_ours & _cand).sum())}")
print(f"is_GAD {int(_gad.sum())} of which {int((_gad & ~_neu).sum())} are NON-neurons; "
      f"is_Interneuron {int(_int.sum())} of which {int((_int & ~_neu).sum())} are "
      f"NON-neurons  [neither flag is neuron-restricted in v4]")

# ── one mutually exclusive label per cell ───────────────────
# No "VH unknown" bucket is needed for the v4 nulls -- there are none. Two
# buckets exist for the mask-less sections because their neurons are genuinely
# unclassifiable with respect to the ventral horn, and calling them "outside VH"
# would be a claim the data does not support. GAD+ non-neurons get their own
# bucket rather than vanishing into "non-neuron": in v4 that is 455 of 474 GAD+
# cells cohort-wide, and folding them in would throw the GAD information away.
_cls = _np.full(adata.n_obs, "non-neuron", dtype=object)
_cls[~_neu & _gad] = "non-neuron, GAD+"
if VH_MASK_PRESENT:
    _cls[_neu & ~_in & ~_gad] = "neuron, outside VH"
    _cls[_neu & ~_in & _gad] = "GABAergic neuron, outside VH"
    _cls[_neu & _in & _gad] = "GABAergic neuron, in VH"
    _cls[_neu & _in & ~_gad & ~_big] = "neuron, in VH (sub-threshold)"
    _cls[_neu & _in & ~_gad & _big] = "MN"
else:
    _cls[_neu & ~_gad] = "neuron, VH mask absent"
    _cls[_neu & _gad] = "GABAergic neuron, VH mask absent"
adata.obs["neuron_class"] = _pd.Categorical(
    _cls, categories=["MN", "neuron, in VH (sub-threshold)",
                      "GABAergic neuron, in VH", "neuron, outside VH",
                      "GABAergic neuron, outside VH", "neuron, VH mask absent",
                      "GABAergic neuron, VH mask absent", "non-neuron, GAD+",
                      "non-neuron"])
assert int(adata.obs["neuron_class"].isna().sum()) == 0, \
    "neuron_class has cells outside its category list -- the ladder is not total"
# The ladder's "MN" bucket is the AREA-GATED call, so it must be checked against
# our obs['is_MN'], NOT against the as-delivered column: v6's is_MN is ungated, so
# comparing to _mn_raw would fire on every section with a sub-threshold candidate.
_cls_mn = int((adata.obs["neuron_class"] == "MN").sum())
_our_mn = int(_pd.Series(adata.obs["is_MN"]).astype("boolean").fillna(False).sum())
assert _cls_mn == _our_mn, (
    f"neuron_class MN ({_cls_mn}) disagrees with obs['is_MN'] ({_our_mn})")
# and the two in-VH non-GAD buckets together must reconstitute the ungated
# candidate, which is what ties the ladder back to Marcel's delivery
if VH_MASK_PRESENT:
    _cls_cand = int(((adata.obs["neuron_class"] == "MN") |
                     (adata.obs["neuron_class"] == "neuron, in VH (sub-threshold)")).sum())
    assert _cls_cand == int(_mn_raw.sum()), (
        f"MN + sub-threshold ({_cls_cand}) != ungated v6 is_MN "
        f"({int(_mn_raw.sum())}); the ladder has drifted from the delivery")

# ── pathologist ground truth, raw counts only ───────────────
# The 183 labels almost certainly trained the v3/v4 classifier (is_neuron_v3
# scores 183/183; all 14 disagreements with v2 resolve in v4's favour and 0
# against) and training membership could not be verified from any file on the
# cluster. They are 0.00887% of the file with a neuronal prevalence 129-fold
# above it. So: raw tp/fn/fp/tn, no percentages, no "sensitivity" anywhere a
# figure could pick it up. The 40 labels on decoy SD02022 cannot reach here
# because the slice is taken by PARQUET_KEY.
_lab = adata.obs["pathologist_class_mw"].to_numpy(dtype=object)
_lab_neuronal = (_lab == "neuronal")
_lab_glial = (_lab == "glial")
_path = {"n_labelled": int(_lab_neuronal.sum() + _lab_glial.sum()),
         "tp_neuronal_called_neuron": int((_lab_neuronal & _neu).sum()),
         "fn_neuronal_called_glia": int((_lab_neuronal & ~_neu).sum()),
         "fp_glial_called_neuron": int((_lab_glial & _neu).sum()),
         "tn_glial_called_glia": int((_lab_glial & ~_neu).sum()),
         "caveat": ("183 hand labels cohort-wide, unevenly spread, 129-fold "
                    "enriched for neurons relative to the file. TRAINING-SET "
                    "MEMBERSHIP NOT VERIFIED -- treat as in-sample. Usable to "
                    "SCORE is_neuron_v3, not as a training or balancing set, "
                    "and never as a generalisation estimate.")}
if _path["n_labelled"]:
    print(f"pathologist_class vs is_neuron_v3 (raw counts, this section): "
          f"TP {_path['tp_neuronal_called_neuron']} "
          f"FN {_path['fn_neuronal_called_glia']} "
          f"FP {_path['fp_glial_called_neuron']} "
          f"TN {_path['tn_glial_called_glia']}  [in-sample, do not quote as a rate]")

# ── sample-level metadata and status, plain strings only ────
# A None anywhere in .uns makes the h5ad unreadable in every anndata version.
# The "unassigned" fallback is load-bearing: cord_level is 100% null on 4 of the
# 22 samples.
adata.uns["sample_meta_v4"] = {
    _k: (str(_m[_k].dropna().unique()[0]) if _m[_k].notna().any() else "unassigned")
    for _k in ("cord_level", "disease_group", "patient_id", "vh_source")}

_n_neu = int(_neu.sum())
_n_mn = int(_mn_raw.sum())
_n_elig = int(adata.obs["is_MN_eligible"].sum())
_n_undet = _n_elig if not VH_MASK_PRESENT else 0
_n_invh = int((_neu & _in).sum())
_n_sub = int((_neu & _in & ~_gad & ~_big).sum())
_n_gadvh = int((_gad & _in).sum())
_n_tdp = int(adata.obs["is_TDP"].sum())
_n_tdp_eval = int(adata.obs["tdp_evaluated"].sum())
_n_area_zero = int((_area <= 0).sum())

adata.uns["vh_status"] = VH_STATUS
adata.uns["mn_call_valid"] = bool(VH_MASK_PRESENT)
adata.uns["mn_call_note"] = (
    "motor-neuron call is valid for this section" if VH_MASK_PRESENT else
    (f"no GM or VH mask for this section: the v4 parquet ships is_MN=False for "
     f"all {adata.n_obs} cells and vh_source="
     f"{adata.uns['sample_meta_v4']['vh_source']}, but 0 cells are in_GM and 0 "
     f"in_VH, so is_MN=0 is a MISSING MASK, not a count. {_n_undet} MN-eligible "
     f"cells are UNDETERMINED. is_MN is <NA> for every cell here on purpose; "
     f"never render this section as 0 motor neurons."))
adata.uns["vh_mask_metrics_v4"] = {
    "n_in_GM": _n_in_gm, "n_in_VH": _n_in_vh,
    "n_in_VH_not_GM": _n_vh_not_gm, "n_MN_outside_GM": _n_mn_outside_gm,
    "n_mn_eligible": _n_elig, "n_mn_undetermined": _n_undet}
adata.uns["pathologist_scores_v4"] = _path
adata.uns["is_TDP_caveat"] = (
    "tdp_proba is exactly 0.0 for all 908,578 Control cells (max 0.0, zero "
    "nulls) and NaN for 281,859 ALS cells, so is_TDP is True in 0 of 10 "
    "controls BY CONSTRUCTION and False also means not-evaluated. Use "
    "tdp_evaluated as the denominator. Keep is_TDP out of every ALS-vs-control "
    "contrast and figure until Marcel explains the fit.")
adata.uns["pathologist_class_caveat"] = _path["caveat"]
adata.uns["in_VH_not_GM_note"] = (
    f"{_n_vh_not_gm} cells are in_VH and not in_GM in this section "
    f"({_n_mn_outside_gm} of them called MN). v4 broke the subset relation on "
    f"7,763 cells cohort-wide, all in robin_pathologist samples. in_GM is NOT "
    f"applied to the MN call; the verified recipe does not use it.")
adata.uns["cell_type_mw_note"] = (
    "cell_type_mw is v4's cell_type, renamed. It is fully derived: "
    "cell_type_mw == 'motor_neuron' equals is_MN and 'other_neuron' equals "
    "is_neuron_v3 & ~is_MN, exact on all 22 sample keys. Its 'glia' bucket is "
    "every non-neuron, endothelium and meninges included. obs['cell_type'] is "
    "this notebook's own marker-gene argmax and is a different thing.")
adata.uns["v6_parquet"] = VH_PARQUET
adata.uns["v6_parquet_md5"] = VH_PARQUET_MD5
adata.uns["relabel_version"] = "v6"

print(f"is_neuron_v3 {_n_neu:,} (stale obs is_neuron was {_n_prev:,}, dropped "
      f"{_dropped}; our markers+size call {_n_gen:,})  ->  in VH {_n_invh:,}  ->  "
      f"is_MN {_n_mn:,}")
print(f"  in-VH GAD+ {_n_gadvh:,} excluded, {_n_sub:,} excluded as < {AREA_MIN} um2; "
      f"MN-eligible ignoring VH {_n_elig:,}")
print(f"  is_TDP {_n_tdp:,} of {_n_tdp_eval:,} evaluated cells "
      f"({adata.n_obs - _n_tdp_eval:,} have tdp_proba NaN)  "
      f"[confounded with disease -- carried, not for contrasts]")
print(f"  cell_area_um2 == 0 on {_n_area_zero:,} cells (2.3% cohort-wide); never "
      f"take log(area) or an area-normalised density without masking area > 0")
print(f"  sample_meta_v4 {adata.uns['sample_meta_v4']}")
print(f"  vh_status {VH_STATUS}; in_GM {_n_in_gm:,}, in_VH {_n_in_vh:,}")
if not VH_MASK_PRESENT:
    print("NO VENTRAL-HORN MASK FOR THIS SECTION -- and no grey-matter mask "
          "either, so there is no geometry to have landed off-tissue.")
    print(f"  is_MN is <NA> for all {adata.n_obs:,} cells BY DESIGN. "
          f"{_n_undet} MN-eligible cells (is_neuron_v3 & ~is_GAD & area >= "
          f"{AREA_MIN}) are UNDETERMINED, not zero.")
    print("  vh_source says 'auto_density_fixed', which is a PLAN tag with no "
          "outcome semantics. Six of the eight auto_density_fixed sections have "
          "a real mask; this one does not.")
print(adata.obs["neuron_class"].value_counts()
      .reindex(adata.obs["neuron_class"].cat.categories).to_string())

try:
    __TRACE__
except NameError:
    __TRACE__ = {}
__TRACE__.update({
    "vh_relabel": True,
    "vh_relabel_version": "v6",
    "vh_parquet": VH_PARQUET,
    "vh_parquet_md5": VH_PARQUET_MD5,
    "vh_parquet_sample": VH_PARQUET_SAMPLE,
    "coord_gate_max_delta_um": _maxd,
    "mn_definition": ("parquet v4 is_MN = is_neuron_v3 & ~is_GAD & in_VH & "
                      f"cell_area_um2 >= {AREA_MIN}"),
    "mn_area_min_um2": AREA_MIN,
    "mn_area_window_um2": "(249.0152, 251.5203] -- 250 is inside it, not unique",
    "n_parquet_dup_rows_dropped": _n_dup,
    "parquet_nulls": _nulls,
    "dropped_stale_cols": list(_dropped),
    "n_is_neuron_general_raw": _n_gen,
    "n_is_neuron_v3_raw": _n_neu,
    "n_is_neuron_prev_raw": _n_prev,
    "n_is_MN_invh_raw": _n_invh,
    "n_is_MN_raw": _n_mn,
    "n_is_MN_eligible_raw": _n_elig,
    "n_mn_undetermined": _n_undet,
    "n_subthreshold_invh_raw": _n_sub,
    "n_is_GADvh_raw": _n_gadvh,
    "n_is_GAD_raw": int(_gad.sum()),
    "n_is_GAD_nonneuron_raw": int((_gad & ~_neu).sum()),
    "n_is_GAD_neuron_raw": int((_gad & _neu).sum()),
    "n_is_Interneuron_raw": int(_int.sum()),
    "n_is_Interneuron_nonneuron_raw": int((_int & ~_neu).sum()),
    "n_is_Interneuron_neuron_raw": int((_int & _neu).sum()),
    "n_is_TDP_raw": _n_tdp,
    "n_tdp_evaluated_raw": _n_tdp_eval,
    "n_in_GM_raw": _n_in_gm,
    "n_in_VH_raw": _n_in_vh,
    "n_inVH_not_inGM": _n_vh_not_gm,
    "n_MN_outside_GM": _n_mn_outside_gm,
    "n_cell_area_zero": _n_area_zero,
    "vh_mask_present": bool(VH_MASK_PRESENT),
    "vh_status": VH_STATUS,
    "mn_call_valid": bool(VH_MASK_PRESENT),
    "has_vh_data": bool(VH_MASK_PRESENT),
    "pathologist_scores_v4": {_k: _v for _k, _v in _path.items() if _k != "caveat"},
    "sample_meta_v4": dict(adata.uns["sample_meta_v4"]),
    "neuron_class_raw": {str(_k): int(_v) for _k, _v in
                         adata.obs["neuron_class"].value_counts().items()},
})
# sentinel: the trace cell asserts this, so a relabel that died half-way cannot
# write a trace that looks clean and get promoted
__RELABEL_V4REAL_OK__ = True


In [ ]:
# Sanity check that the object holds the one sample it should.

adata.obs["sample"].cat.categories

In [ ]:
# Print the panel so gene names can be checked before markers are used.

print(adata.var_names.tolist())

# Sample SD01923_BI

In [ ]:
# Work on a copy from here, and confirm the centroid columns are present.

sample = adata.copy()

# check what coordinate columns are available
print([c for c in sample.obs.columns if 'centroid' in c.lower() or 'x_' in c.lower() or 'y_' in c.lower()])

## QC metrics

In [ ]:
# First look at the count distributions. The dashed lines are the Salas et
# al. cut-offs of 10 transcripts and 10 genes per cell, drawn but not yet
# applied. Read this before choosing the thresholds below.

sc.pp.calculate_qc_metrics(sample, percent_top=[20, 50, 100, 200], log1p=False, inplace=True)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(sample.obs['total_counts'], bins=100, color='#1A56DB', edgecolor='none')
axes[0].axvline(10, color='#2D3748', linestyle='--', label='Salas et al. (10 Tx)')
axes[0].set_xlabel('Total transcripts per cell'); axes[0].set_ylabel('Number of cells')
axes[0].set_title('Transcript count distribution'); axes[0].legend()

axes[1].hist(sample.obs['n_genes_by_counts'], bins=100, color='#276749', edgecolor='none')
axes[1].axvline(10, color='#C53030', linestyle='--', label='QC threshold (10 genes)')
axes[1].set_xlabel('Unique genes per cell'); axes[1].set_title('Gene complexity distribution')
axes[1].legend()

axes[2].hist(np.log1p(sample.obs['total_counts']), bins=100, color='#B7791F', edgecolor='none')
axes[2].axvline(np.log1p(10), color='#2D3748', linestyle='--', label='Salas et al.')
axes[2].set_xlabel('log(1 + total transcripts)'); axes[2].set_title('Log-transformed distribution')
axes[2].legend()

sns.despine(fig=fig)
plt.tight_layout()
plt.savefig(FIGDIR / '(RK)SD01923BI_qc_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Cells before QC: {sample.n_obs:,}")
print(f"Median transcripts per cell: {sample.obs['total_counts'].median():.0f}")
print(f"Cells with < 10 transcripts (Salas): {(sample.obs['total_counts'] < 10).sum():,} ({(sample.obs['total_counts'] < 10).mean()*100:.1f}%)")
print(f"Cells with < 10 genes: {(sample.obs['n_genes_by_counts'] < 10).sum():,} ({(sample.obs['n_genes_by_counts'] < 10).mean()*100:.1f}%)")

## Novae spatial domains

The domains were computed per sample by the independent Novae run and travel
with the h5ad. They are used here as a QC lens: a domain that is really a smear,
a fold or a detachment tends to look nothing like the rest of the tissue in the
count statistics, and it tends to sit in one contiguous blob.


In [ ]:
# Draw the Novae domains that came with the file, one panel per resolution.
# Novae was run at four levels of granularity (n4, n6, n8, n10 domains). Cells
# it could not place are labelled 'unassigned' and are drawn in grey; they
# cluster wherever the neighbourhood graph is thin, which is itself a QC signal.
# The last panel shows neighborhood_valid, Novae's own flag for cells whose
# spatial neighbourhood was too sparse to build a niche from.

DOMAIN_KEYS = [k for k in ['novae_domains_n4', 'novae_domains_n6',
                           'novae_domains_n8', 'novae_domains_n10']
               if k in sample.obs.columns]
print(f"Novae domain columns found: {DOMAIN_KEYS}")

panels = list(DOMAIN_KEYS)
if 'neighborhood_valid' in sample.obs.columns:
    panels.append('neighborhood_valid')

ncol = 2
nrow = int(np.ceil(len(panels) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(7.5 * ncol, 6.5 * nrow), dpi=200)
axes = np.atleast_1d(axes).ravel()

x = sample.obs['x_centroid'].values
y = sample.obs['y_centroid'].values

for ax, key in zip(axes, panels):
    vals = sample.obs[key]

    if key == 'neighborhood_valid':
        ok = np.asarray(vals).astype(bool)
        ax.scatter(x[ok], y[ok], s=0.4, c='#1A56DB', edgecolors='none',
                   rasterized=True, label=f'valid (n={ok.sum():,})')
        ax.scatter(x[~ok], y[~ok], s=1.2, c='#C53030', edgecolors='none',
                   rasterized=True, label=f'sparse (n={(~ok).sum():,})')
        ax.set_title('Novae neighbourhood validity', fontsize=10,
                     fontweight='bold', pad=8)
        ax.legend(fontsize=7, frameon=False, markerscale=6, loc='upper right')
    else:
        cats = list(vals.cat.categories) if hasattr(vals, 'cat') else sorted(vals.unique())
        # 'unassigned' is always grey; everything else takes a tab20 colour.
        real = [c for c in cats if c != 'unassigned']
        cmap = plt.cm.tab20(np.linspace(0, 1, max(len(real), 1)))
        colour = {c: cmap[i] for i, c in enumerate(real)}
        colour['unassigned'] = (0.82, 0.82, 0.82, 1.0)

        for c in cats:
            m = (vals == c).values
            if not m.any():
                continue
            ax.scatter(x[m], y[m], s=0.4, c=[colour[c]], edgecolors='none',
                       rasterized=True, label=f'{c} ({m.sum():,})')
        ax.set_title(f"{key} ({len(real)} domains)", fontsize=10,
                     fontweight='bold', pad=8)
        ax.legend(fontsize=6, frameon=False, markerscale=6, ncol=1,
                  loc='upper left', bbox_to_anchor=(1.01, 1.0))

    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.axis('off')

for ax in axes[len(panels):]:
    ax.axis('off')

plt.tight_layout()
plt.savefig(FIGDIR / '(RK)SD01923BI_novae_domains_spatial.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(FIGDIR / '(RK)SD01923BI_novae_domains_spatial.pdf',
            bbox_inches='tight', facecolor='white')
plt.show()

for key in DOMAIN_KEYS:
    counts = sample.obs[key].value_counts()
    print(f"\n{key}")
    print((counts / counts.sum() * 100).round(2).to_string())


In [ ]:
# Turn the Novae domains into a QC table.
#
# The idea is simple. A domain that is tissue should look like tissue: decent
# transcript counts, a sensible number of genes, cell areas in the usual range,
# a small share of negative-control counts and neighbours all around it. A
# domain that is a smear, a fold, a detachment or an empty-field artefact fails
# several of those at once, and it fails them together in one place on the slide.
#
# Each metric below is turned into a z-score across domains (sign flipped where
# low is bad), and the mean of those z-scores becomes suspicion_score. High
# score means "look at this domain before you trust it". It is a ranking aid,
# not a verdict, so read it next to the maps in the previous cell.

DOMAIN_QC_KEY = 'novae_domains_n8'      # resolution used for the table
COMPUTE_NEIGHBOUR_DENSITY = True        # local density, the strongest smear cue
DENSITY_RADIUS_UM = 100

assert DOMAIN_QC_KEY in sample.obs.columns, f"{DOMAIN_QC_KEY} not in obs"

# QC metrics may not exist yet if this cell is run out of order.
if 'total_counts' not in sample.obs.columns or 'n_genes_by_counts' not in sample.obs.columns:
    sc.pp.calculate_qc_metrics(sample, percent_top=[20, 50, 100, 200],
                               log1p=False, inplace=True)

obs = sample.obs

# Share of each cell's counts that landed on negative controls rather than on
# real probes. Elevated in debris and in regions with poor optical quality.
neg_cols = [c for c in ['control_probe_counts', 'control_codeword_counts',
                        'genomic_control_counts', 'unassigned_codeword_counts',
                        'deprecated_codeword_counts'] if c in obs.columns]
if neg_cols:
    neg_total = obs[neg_cols].sum(axis=1)
    obs['neg_control_frac'] = neg_total / (obs['total_counts'] + neg_total).clip(lower=1)
else:
    obs['neg_control_frac'] = np.nan

# Nucleus to cell area ratio. Segmentation that has run away from the nucleus
# gives tiny ratios; cells with no nucleus at all give zero.
if {'nucleus_area', 'cell_area'}.issubset(obs.columns):
    obs['nucleus_ratio'] = obs['nucleus_area'] / obs['cell_area'].clip(lower=1e-9)
else:
    obs['nucleus_ratio'] = np.nan

# Neighbour count inside DENSITY_RADIUS_UM. Written to its own key so it does
# not tread on the spatial_100um graph the filter cells build later.
if COMPUTE_NEIGHBOUR_DENSITY:
    sq.gr.spatial_neighbors(sample, coord_type='generic',
                            radius=DENSITY_RADIUS_UM, key_added='novae_qc_density')
    obs['n_neighbours_qc'] = np.asarray(
        sample.obsp['novae_qc_density_connectivities'].sum(axis=1)).ravel()
else:
    obs['n_neighbours_qc'] = np.nan

agg = {
    'n_cells':          ('total_counts', 'size'),
    'median_counts':    ('total_counts', 'median'),
    'median_genes':     ('n_genes_by_counts', 'median'),
    'pct_lt10_counts':  ('total_counts', lambda s: (s < 10).mean() * 100),
    'pct_lt10_genes':   ('n_genes_by_counts', lambda s: (s < 10).mean() * 100),
    'median_neg_frac':  ('neg_control_frac', 'median'),
    'median_neighbours': ('n_neighbours_qc', 'median'),
}
if 'cell_area' in obs.columns:
    agg['median_cell_area'] = ('cell_area', 'median')
if 'nucleus_ratio' in obs.columns:
    agg['median_nucleus_ratio'] = ('nucleus_ratio', 'median')
if 'nucleus_count' in obs.columns:
    agg['pct_no_nucleus'] = ('nucleus_count', lambda s: (s == 0).mean() * 100)

domain_qc = obs.groupby(DOMAIN_QC_KEY, observed=True).agg(**agg)
domain_qc['pct_of_cells'] = domain_qc['n_cells'] / domain_qc['n_cells'].sum() * 100

# Fraction of each domain that Novae itself was unhappy about.
if 'neighborhood_valid' in obs.columns:
    domain_qc['pct_sparse_nbhd'] = (
        obs.groupby(DOMAIN_QC_KEY, observed=True)['neighborhood_valid']
           .apply(lambda s: (~s.astype(bool)).mean() * 100))

# Spatial compactness. sd_xy is the mean spread of a domain's cells around its
# own centre, scaled by the spread of the whole section. Values near 1 mean the
# domain is sprinkled over the section (usually a cell type). Small values mean
# it sits in one patch (a real anatomical region, or an artefact).
span = np.hypot(obs['x_centroid'].std(), obs['y_centroid'].std())
compact = obs.groupby(DOMAIN_QC_KEY, observed=True).apply(
    lambda d: np.hypot(d['x_centroid'].std(), d['y_centroid'].std()) / span)
domain_qc['spatial_spread'] = compact

# Composition, when the labels are around. A domain that is nearly all neurons
# or nearly no neurons is worth a second look either way.
for flag in ['is_neuron_v3', 'is_MN']:
    if flag in obs.columns:
        domain_qc[f'pct_{flag}'] = (
            obs.groupby(DOMAIN_QC_KEY, observed=True)[flag]
               .apply(lambda s: pd.Series(s).astype('boolean').fillna(False)
                          .to_numpy(dtype=bool).mean() * 100))

# ---- suspicion score -------------------------------------------------------
def _z(v):
    v = pd.to_numeric(v, errors='coerce')
    sd = v.std(ddof=0)
    return pd.Series(0.0, index=v.index) if not np.isfinite(sd) or sd == 0 else (v - v.mean()) / sd

terms = []
terms.append(-_z(domain_qc['median_counts']))        # few transcripts is bad
terms.append(-_z(domain_qc['median_genes']))         # low complexity is bad
terms.append(_z(domain_qc['pct_lt10_counts']))       # many failing cells is bad
if 'median_neg_frac' in domain_qc:
    terms.append(_z(domain_qc['median_neg_frac']))   # negative controls climbing is bad
if COMPUTE_NEIGHBOUR_DENSITY:
    terms.append(-_z(domain_qc['median_neighbours']))  # isolated cells are bad
if 'median_nucleus_ratio' in domain_qc:
    terms.append(-_z(domain_qc['median_nucleus_ratio']))  # nucleus-free cells are bad
terms.append(-_z(domain_qc['spatial_spread']))       # one tight blob is suspicious

domain_qc['suspicion_score'] = pd.concat(terms, axis=1).mean(axis=1)
domain_qc = domain_qc.sort_values('suspicion_score', ascending=False)

cols = ['n_cells', 'pct_of_cells', 'median_counts', 'median_genes',
        'pct_lt10_counts', 'median_neighbours', 'median_neg_frac',
        'spatial_spread', 'suspicion_score']
cols = [c for c in cols if c in domain_qc.columns]
print(f"=== Novae domain QC — SD01923_BI — {DOMAIN_QC_KEY} ===")
print(domain_qc[cols].round(3).to_string())

domain_qc.round(4).to_csv(FIGDIR / '(RK)SD01923BI_novae_domain_qc.csv')

# ---- how the metrics look side by side -------------------------------------
heat_cols = [c for c in ['median_counts', 'median_genes', 'pct_lt10_counts',
                         'median_neighbours', 'median_neg_frac',
                         'median_cell_area', 'median_nucleus_ratio',
                         'pct_no_nucleus', 'spatial_spread', 'pct_sparse_nbhd']
             if c in domain_qc.columns]
heat = domain_qc[heat_cols].apply(_z)

fig, axes = plt.subplots(1, 2, figsize=(6 + 0.9 * len(heat_cols), 0.45 * len(domain_qc) + 3),
                         dpi=200, gridspec_kw={'width_ratios': [3, 1]})

sns.heatmap(heat, cmap='RdBu_r', center=0, ax=axes[0],
            cbar_kws={'label': 'z across domains', 'shrink': 0.6},
            linewidths=0.4, linecolor='white')
axes[0].set_title(f'Novae domain QC metrics ({DOMAIN_QC_KEY})',
                  fontsize=10, fontweight='bold', pad=8)
axes[0].set_ylabel('')
axes[0].tick_params(labelsize=7)
plt.setp(axes[0].get_xticklabels(), rotation=40, ha='right')

axes[1].barh(range(len(domain_qc)), domain_qc['suspicion_score'].values,
             color=['#C53030' if v > 0.5 else '#94A3B8'
                    for v in domain_qc['suspicion_score']])
axes[1].set_yticks(range(len(domain_qc)))
axes[1].set_yticklabels(domain_qc.index, fontsize=7)
axes[1].invert_yaxis()
axes[1].axvline(0.5, color='#C53030', linestyle='--', linewidth=0.8)
axes[1].set_title('Suspicion score', fontsize=10, fontweight='bold', pad=8)
axes[1].tick_params(labelsize=7)
sns.despine(ax=axes[1])

plt.tight_layout()
plt.savefig(FIGDIR / '(RK)SD01923BI_novae_domain_qc.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(FIGDIR / '(RK)SD01923BI_novae_domain_qc.pdf',
            bbox_inches='tight', facecolor='white')
plt.show()

flagged = domain_qc.index[domain_qc['suspicion_score'] > 0.5].tolist()
print(f"\nDomains above the 0.5 line: {flagged}")
print("Check them on the maps above before doing anything with them. "
      "To drop one:  sample = sample[~sample.obs[DOMAIN_QC_KEY].isin(flagged)].copy()")

# How stable the labels are across granularities. A domain that survives intact
# from n4 through n10 is a solid structure. One that shatters is a soft call.
other = [k for k in DOMAIN_KEYS if k != DOMAIN_QC_KEY]
if other:
    ref = obs[DOMAIN_QC_KEY].astype(str)
    for k in other:
        ct = pd.crosstab(ref, obs[k].astype(str), normalize='index') * 100
        purity = ct.max(axis=1)
        print(f"\n{DOMAIN_QC_KEY} rows split across {k} columns (% of row)")
        print(ct.round(1).to_string())
        print(f"largest single overlap per domain:\n{purity.round(1).to_string()}")


In [ ]:
# The same QC metrics painted on tissue coordinates. Low-count patches
# that follow tissue structure are biology. Low-count patches that follow
# a stripe or a blob are usually a smear or an out-of-focus region.

sc.pp.calculate_qc_metrics(sample, percent_top=[20, 50, 100, 200], log1p=False, inplace=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = [
    ('total_counts', 'viridis', 'Total transcripts', True),
    ('n_genes_by_counts', 'magma', 'Unique genes', False),
    ('log1p_total_over_genes', 'RdYlBu_r', 'Transcripts per gene', False),
]

sample.obs['log1p_total_over_genes'] = np.log1p(
    sample.obs['total_counts'] / sample.obs['n_genes_by_counts'].clip(lower=1)
)

for ax, (col, cmap, label, use_log) in zip(axes, metrics):
    vals = np.asarray(sample.obs[col]).ravel()
    
    if use_log:
        vals = np.log1p(vals)
        label = f'{label} (log1p)'
    
    vmin, vmax = np.percentile(vals, [2, 98])
    
    scat = ax.scatter(sample.obs['x_centroid'], sample.obs['y_centroid'],
                      c=vals, s=0.3, cmap=cmap, rasterized=True,
                      vmin=vmin, vmax=vmax)
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.set_title(f'SD01923_BI — {label}')
    sns.despine(ax=ax, left=True, bottom=True)
    ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(scat, ax=ax, label=label, shrink=0.8)

plt.tight_layout()
plt.savefig(FIGDIR / '(RK)SD01923BI_spatial_qc.png', dpi=200, bbox_inches='tight')
plt.show()

## Transcript filter

In [ ]:
# Apply the transcript cut-off and redraw the tissue, so it is obvious
# whether the filter ate a real anatomical region.

MIN_TRANSCRIPTS = 10
n_before = sample.n_obs
sample = sample[sample.obs['total_counts'] >= MIN_TRANSCRIPTS].copy()
print(f"After transcript filter (>= {MIN_TRANSCRIPTS}): {sample.n_obs:,} cells ({sample.n_obs/n_before*100:.1f}%)")

fig, ax = plt.subplots(figsize=(7, 6), dpi=200)
ax.scatter(sample.obs['x_centroid'], sample.obs['y_centroid'],
           s=2, alpha=0.5, c='#1A56DB', edgecolors='none', rasterized=True)
ax.set_title(f'After transcript filter (n={sample.n_obs:,})', fontsize=10, fontweight='bold', pad=8)
ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')
plt.tight_layout()
plt.savefig(FIGDIR / '(RK)SD01923BI_post_transcript_filter.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(FIGDIR / '(RK)SD01923BI_post_transcript_filter.pdf', bbox_inches='tight', facecolor='white')

plt.show()

## Spatial neighbours filter preview

In [ ]:
# Preview of the density filter. Cells with fewer than 8 neighbours inside
# 100 um are flagged in red. Nothing is removed yet. Isolated cells at the
# tissue edge or floating in the empty background show up here.

sq.gr.spatial_neighbors(sample, coord_type='generic', radius=100, key_added='spatial_100um')
A = sample.obsp['spatial_100um_connectivities']
sample.obs['n_spatial_neighbours_100um'] = np.array(A.sum(axis=1)).flatten()

fig, ax = plt.subplots(figsize=(7, 6))
removed = sample.obs['n_spatial_neighbours_100um'] < 8
ax.scatter(sample.obs.loc[~removed, 'x_centroid'], sample.obs.loc[~removed, 'y_centroid'],
           s=0.3, c='#1A56DB', rasterized=True, label='Kept')
ax.scatter(sample.obs.loc[removed, 'x_centroid'], sample.obs.loc[removed, 'y_centroid'],
           s=1, c='#C53030', rasterized=True, label='Removed')
ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')
ax.legend(markerscale=5)
plt.tight_layout()
plt.savefig(FIGDIR / '(RK)SD01923BI_spatial_filter_preview.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(FIGDIR / '(RK)SD01923BI_spatial_filter_preview.pdf', bbox_inches='tight', facecolor='white')
plt.show()
print(f"Would remove: {removed.sum():,} cells ({removed.mean()*100:.1f}%)")

## Spatial filter real run

In [ ]:
# Apply the density filter.

MIN_NEIGHBOURS = 8
n_before = sample.n_obs
sample = sample[sample.obs['n_spatial_neighbours_100um'] >= MIN_NEIGHBOURS].copy()
print(f"After spatial filter (>= {MIN_NEIGHBOURS} neighbours in 100µm): {sample.n_obs:,} cells ({sample.n_obs/n_before*100:.1f}%)")

## QC decision trace

Auto-generated summary of every filtering decision made above. Each threshold below is tagged `[Salas et al.]` when it is the published benchmark from that paper, or `[pipeline]` when it is a heuristic specific to this dual-pass design (the Novae domain suspicion screen and the 100um neighbour-density filter are not in the Salas et al. table).

In [ ]:
# QC decision trace. Pulls together the domain suspicion screen and the two
# hard filters above into one auditable record, in the style used for the
# manuscript's methods QC log. No operator field: everything here is
# recomputed from the objects already in memory, not typed in by hand.

from datetime import datetime

sample_label = str(adata.obs["sample"].cat.categories[0])

raw_n = adata.n_obs
if "n_genes_by_counts" not in adata.obs.columns:
    sc.pp.calculate_qc_metrics(adata, percent_top=[20, 50, 100, 200], log1p=False, inplace=True)

neg_cols = [c for c in ["control_probe_counts", "control_codeword_counts",
                        "genomic_control_counts", "unassigned_codeword_counts",
                        "deprecated_codeword_counts"] if c in adata.obs.columns]
if neg_cols:
    neg_total = adata.obs[neg_cols].sum(axis=1)
    neg_frac_median = float((neg_total / (adata.obs["total_counts"] + neg_total).clip(lower=1)).median())
    snr_aggregate = float(adata.obs["total_counts"].sum() / max(neg_total.sum(), 1))
else:
    neg_frac_median, snr_aggregate = float("nan"), float("nan")

# n_before was set two cells above, right before the spatial filter ran, so it
# holds the post-transcript-filter / pre-spatial-filter count. sample.n_obs is
# the count after that filter, i.e. right now.
n_after_transcript = n_before
n_after_spatial = sample.n_obs

removed_transcript = raw_n - n_after_transcript
pct_removed_transcript = removed_transcript / raw_n * 100
removed_spatial = n_after_transcript - n_after_spatial
pct_removed_spatial = removed_spatial / n_after_transcript * 100

n_domains_scored = domain_qc.shape[0] if "domain_qc" in globals() else None
flagged_domains = flagged if "flagged" in globals() else []
mn_note = ""
if flagged_domains and "domain_qc" in globals() and "pct_is_MN" in domain_qc.columns:
    _mn_base = float(domain_qc["pct_is_MN"].median())
    # under v4 that median is exactly 0.0 in 16 of 20 sections, which makes
    # "above median" true for any domain holding a single motor neuron. Say
    # which test actually applied rather than implying a distribution.
    hot = [d for d in flagged_domains if domain_qc.loc[d, "pct_is_MN"] > _mn_base]
    if hot:
        _how = (f"above the {_mn_base:.3f}% domain median" if _mn_base > 0
                else "any motor neurons at all (domain median is 0.000%)")
        mn_note = f" ({', '.join(hot)} carry {_how} -- check MN somata before calling these smear)"

transcript_verdict = ("PASS" if pct_removed_transcript < 1 else
                      "REVIEW (above the Salas et al. 0.21% benchmark; "
                      "some elevation is expected in mixed grey/white matter cord)")
spatial_verdict = ("PASS (retention in the typical 90-98% range for intact tissue)"
                    if pct_removed_spatial <= 10 else
                    "REVIEW (>10% removed -- check the tissue plot above for whether this "
                    "tracks white-matter sparsity [biological] or a technical dropout; "
                    "not an automatic fail either way)")
neg_verdict = ("PASS" if neg_cols and neg_frac_median < 0.01 and snr_aggregate > 100
               else ("N/A (no negative-control columns on this object)" if not neg_cols else "REVIEW"))

conclusion = ("PASS. Proceed to clustering."
              if transcript_verdict == "PASS" and spatial_verdict.startswith("PASS")
              else "REVIEW before clustering -- see the flagged steps above.")

print(f"""=== QC DECISION TRACE: {sample_label} ===
Date: {datetime.now():%Y-%m-%d}
Software: scanpy {sc.__version__}, squidpy {sq.__version__}

RAW DATA:
  Cells loaded: {raw_n:,}
  Median transcripts/cell: {adata.obs['total_counts'].median():.0f}
  Median genes/cell: {adata.obs['n_genes_by_counts'].median():.0f}
  Negative-control fraction (median/cell): {neg_frac_median:.4f}  [Salas et al. threshold <0.01]
  Negative-control SNR (aggregate): {snr_aggregate:.0f}x  [Salas et al. threshold >100x] -> {neg_verdict}

CELL FILTERING:
  Step 1: Novae domain suspicion screen (novae_domains_n8)  [pipeline]
    Domains scored: {n_domains_scored}
    Flagged (suspicion_score > 0.5): {', '.join(flagged_domains) if flagged_domains else 'none'}{mn_note}
    Decision: held for visual review, carried forward to the post-clustering smear pass
              further down this notebook -- not dropped at this stage

  Step 2: Transcript-count filter  [Salas et al.]
    Threshold: total_counts >= {MIN_TRANSCRIPTS}
    Before: {raw_n:,} cells
    After:  {n_after_transcript:,} cells ({100 - pct_removed_transcript:.1f}% retained, {removed_transcript:,} removed)
    Decision: {transcript_verdict}

  Step 3: Spatial-neighbour density filter  [pipeline]
    Threshold: >= {MIN_NEIGHBOURS} neighbours within 100um
    Before: {n_after_transcript:,} cells
    After:  {n_after_spatial:,} cells ({100 - pct_removed_spatial:.1f}% retained, {removed_spatial:,} removed)
    Decision: {spatial_verdict}

FINAL DATASET: {n_after_spatial:,} cells; {adata.n_vars} genes ({n_after_spatial / raw_n * 100:.1f}% of loaded cells retained overall)
CONCLUSION: {conclusion}
NOTE: post-mortem RNA integrity (RIN/PMI) is not assessed at this step. The Salas et al.
thresholds above were calibrated on non-CNS, non-post-mortem tissue and may be optimistic
for this donor class; read a PASS here as "consistent with that benchmark", not as a
guarantee of equivalent quality.""")


## Normalisation - target_sum = 100 from Salas

In [ ]:
# Normalise to 100 counts per cell and log-transform, following Salas et
# al. for Xenium-scale panels. STMN2 is plotted afterwards as a spot check
# that the ventral horn still looks like a ventral horn.

sample.layers['counts'] = sample.X.copy()
sc.pp.normalize_total(sample, target_sum=100)
sc.pp.log1p(sample)
sample.layers['log1p_norm'] = sample.X.copy()

if 'STMN2' in sample.var_names:
    expr = np.asarray(sample[:, 'STMN2'].X.todense()).flatten() if sp.issparse(sample.X) else sample[:, 'STMN2'].X.flatten()
    vmin, vmax = np.percentile(expr, [1, 99])

    fig, ax = plt.subplots(figsize=(7, 6), dpi=200)
    sc_plot = ax.scatter(
        sample.obs['x_centroid'], sample.obs['y_centroid'],
        c=expr, s=2, cmap='viridis',
        norm=mcolors.Normalize(vmin=vmin, vmax=vmax),
        edgecolors='none', rasterized=True
    )
    ax.set_title('STMN2 — post-normalisation', fontsize=10, fontstyle='italic', fontweight='bold', pad=8)
    ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')

    cbar = fig.colorbar(sc_plot, ax=ax, fraction=0.046, pad=0.04, shrink=0.8)
    cbar.set_label('log1p normalised expression', fontsize=8)
    cbar.outline.set_visible(False)
    cbar.ax.tick_params(labelsize=7, length=2, width=0.5)

    plt.tight_layout()
    plt.savefig(FIGDIR / '(RK)SD01923BI_stmn2_normalised.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(FIGDIR / '(RK)SD01923BI_stmn2_normalised.pdf', bbox_inches='tight', facecolor='white')
    plt.show()
else:
    print(f"STMN2 not in panel. Available similar: {[g for g in sample.var_names if 'STM' in g.upper()]}")

print(f"Normalisation complete. Layers: {list(sample.layers.keys())}")

## No feature selection -- straight to dimentionality reduction, PCA

In [ ]:
# Scale and run PCA on all genes. A 480-gene panel has no HVG step to do.
# The geometric elbow is drawn for reference, but pass 1 keeps all 50 PCs.

# ---- Scale + PCA (all genes, no HVG selection per Salas) ----
sc.pp.scale(sample)
sc.tl.pca(sample, n_comps=50, svd_solver='arpack')

variance_ratio = sample.uns['pca']['variance_ratio']
cumulative      = np.cumsum(variance_ratio)

# Geometric elbow
xy     = np.column_stack([np.arange(50), variance_ratio])
d      = xy[-1] - xy[0]
d_norm = d / np.linalg.norm(d)
dists  = np.abs(np.cross(d_norm, xy - xy[0]))
ELBOW_PC = int(np.argmax(dists)) + 1

# Salas et al.: use all PCs. Show elbow for reference.
N_PCS = 50

# ---- Scree + cumulative variance ----
fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=200)

ax = axes[0]
ax.plot(range(1, 51), variance_ratio, 'o', color='#1A56DB', markersize=3.5,
        markeredgecolor='none', zorder=2)
ax.plot(range(1, 51), variance_ratio, '-', color='#1A56DB', linewidth=0.8,
        alpha=0.4, zorder=1)
ax.axvline(ELBOW_PC, color='#2D6A4F', linestyle='--', linewidth=0.9,
           label=f'Geometric elbow (PC{ELBOW_PC})', zorder=3)
ax.axvline(N_PCS, color='#C53030', linestyle='--', linewidth=0.9,
           label=f'Salas et al. (all PCs)', zorder=3)
ax.fill_between(range(1, N_PCS + 1), variance_ratio[:N_PCS],
                color='#1A56DB', alpha=0.07, zorder=0)
ax.set_yscale('log')
ax.set_xlabel('Principal component', fontsize=9)
ax.set_ylabel('Variance ratio (log)', fontsize=9)
ax.set_title('Scree plot', fontsize=10, fontweight='bold', pad=10)
ax.legend(fontsize=7, frameon=False)
ax.tick_params(labelsize=7, length=2, width=0.5)
for s in ['top', 'right']:   ax.spines[s].set_visible(False)
for s in ['bottom', 'left']: ax.spines[s].set_linewidth(0.5)
ax.text(-0.1, 1.05, 'a', transform=ax.transAxes, fontsize=14, fontweight='bold', va='top')

ax = axes[1]
ax.plot(range(1, 51), cumulative * 100, '-', color='#276749', linewidth=1.5)
ax.axvline(ELBOW_PC, color='#2D6A4F', linestyle='--', linewidth=0.9,
           label=f'Geometric elbow PC{ELBOW_PC}: {cumulative[ELBOW_PC-1]*100:.1f}%')
ax.axvline(N_PCS, color='#C53030', linestyle='--', linewidth=0.9,
           label=f'Salas et al. PC{N_PCS}: {cumulative[N_PCS-1]*100:.1f}%')
ax.axhline(cumulative[N_PCS-1] * 100, color='#C53030', linestyle=':', linewidth=0.5, alpha=0.5)
ax.set_xlabel('Principal component', fontsize=9)
ax.set_ylabel('Cumulative variance (%)', fontsize=9)
ax.set_title('Cumulative variance', fontsize=10, fontweight='bold', pad=10)
ax.legend(fontsize=7, frameon=False)
ax.tick_params(labelsize=7, length=2, width=0.5)
for s in ['top', 'right']:   ax.spines[s].set_visible(False)
for s in ['bottom', 'left']: ax.spines[s].set_linewidth(0.5)
ax.text(-0.1, 1.05, 'b', transform=ax.transAxes, fontsize=14, fontweight='bold', va='top')

plt.tight_layout()
plt.savefig(FIGDIR / '(RK)SD01923BI_pca_variance.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(FIGDIR / '(RK)SD01923BI_pca_variance.pdf', bbox_inches='tight', facecolor='white')
plt.show()

# ---- Helper: get log-normalised expression ----
def get_lognorm(gene):
    idx   = sample.var_names.get_loc(gene)
    layer = sample.layers['log1p_norm']
    return (np.asarray(layer[:, idx].todense()).flatten()
            if sp.issparse(layer) else layer[:, idx].flatten())

# ---- PCA marker embeddings ----
marker_candidates = ['STMN2', 'MOG', 'OLIG2', 'GFAP', 'RBFOX3', 'WDR49', 'TREM2', 'CD68', 'CD4']
markers_available = [g for g in marker_candidates if g in sample.var_names]
markers_missing   = [g for g in marker_candidates if g not in sample.var_names]
if markers_missing:
    print(f"Not in panel: {markers_missing}")

if markers_available:
    pca_coords = sample.obsm['X_pca'][:, :2]
    ncols = min(4, len(markers_available))
    nrows = int(np.ceil(len(markers_available) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 4 * nrows), dpi=200)
    axes_flat = np.atleast_1d(axes).flatten()

    for i, gene in enumerate(markers_available):
        ax = axes_flat[i]
        expr = get_lognorm(gene)
        vmin, vmax = np.percentile(expr[expr > 0], [1, 99]) if (expr > 0).any() else (0, 1)
        zero_mask = expr <= 0
        ax.scatter(pca_coords[zero_mask, 0], pca_coords[zero_mask, 1],
                   s=1, c='#E8E8E8', edgecolors='none', rasterized=True, zorder=1)
        sc_plot = ax.scatter(
            pca_coords[~zero_mask, 0], pca_coords[~zero_mask, 1],
            c=expr[~zero_mask], cmap='viridis',
            norm=mcolors.Normalize(vmin=vmin, vmax=vmax),
            s=1.5, edgecolors='none', rasterized=True, zorder=2)
        ax.set_title(gene, fontsize=10, fontstyle='italic', fontweight='bold', pad=8)
        ax.axis('off')
        cbar = fig.colorbar(sc_plot, ax=ax, fraction=0.046, pad=0.04, shrink=0.8)
        cbar.outline.set_visible(False)
        cbar.ax.tick_params(labelsize=7, length=2, width=0.5)
        cbar.set_label('log1p norm', fontsize=6)

    for j in range(len(markers_available), len(axes_flat)):
        axes_flat[j].set_visible(False)

    plt.tight_layout()
    plt.savefig(FIGDIR / '(RK)SD01923BI_pca_markers.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(FIGDIR / '(RK)SD01923BI_pca_markers.pdf', bbox_inches='tight', facecolor='white')
    plt.show()

# ---- Neighbours + UMAP (Salas: 16 neighbours, all PCs, Louvain) ----
sc.pp.neighbors(sample, n_neighbors=16, n_pcs=N_PCS)
sc.tl.umap(sample, min_dist=0.3)

umap_markers = [g for g in markers_available]
umap_coords  = sample.obsm['X_umap']
ncols = 4
nrows = int(np.ceil(len(umap_markers) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.8 * nrows), dpi=200)
axes_flat = np.atleast_1d(axes).flatten()

for i, gene in enumerate(umap_markers):
    ax = axes_flat[i]
    expr = get_lognorm(gene)
    vmin, vmax = np.percentile(expr[expr > 0], [1, 99]) if (expr > 0).any() else (0, 1)
    zero_mask = expr <= 0
    ax.scatter(umap_coords[zero_mask, 0], umap_coords[zero_mask, 1],
               s=0.8, c='#E8E8E8', edgecolors='none', rasterized=True, zorder=1)
    sc_plot = ax.scatter(
        umap_coords[~zero_mask, 0], umap_coords[~zero_mask, 1],
        c=expr[~zero_mask], cmap='viridis',
        norm=mcolors.Normalize(vmin=vmin, vmax=vmax),
        s=1.2, edgecolors='none', rasterized=True, zorder=2)
    ax.set_title(gene, fontsize=10, fontstyle='italic', fontweight='bold', pad=6)
    ax.axis('off')
    cbar = fig.colorbar(sc_plot, ax=ax, fraction=0.046, pad=0.04, shrink=0.8)
    cbar.outline.set_visible(False)
    cbar.ax.tick_params(labelsize=6, length=2, width=0.4)

for j in range(len(umap_markers), len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.tight_layout()
plt.savefig(FIGDIR / '(RK)SD01923BI_umap_markers.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(FIGDIR / '(RK)SD01923BI_umap_markers.pdf', bbox_inches='tight', facecolor='white')
plt.show()

print(f"Geometric elbow: PC{ELBOW_PC} ({cumulative[ELBOW_PC-1]*100:.1f}% cumulative variance)")
print(f"Selected N_PCS:  {N_PCS} — Salas et al. (all PCs)")
print(f"Cumulative variance at PC{N_PCS}: {cumulative[N_PCS-1]*100:.1f}%")

## Clustering resolutions

In [ ]:
# Leiden at three resolutions, shown on UMAP and on tissue side by side.
# Resolution 1.0 is what the smear hunt below uses. Resolution 1.5 becomes
# the working clustering.

resolutions = [1.0, 1.2, 1.5]
for res in resolutions:
    sc.tl.leiden(sample, resolution=res, random_state=42, key_added=f'leiden_{res}')
    print(f"Resolution {res}: {sample.obs[f'leiden_{res}'].nunique()} clusters")

umap_coords = sample.obsm['X_umap']

fig, axes = plt.subplots(3, 2, figsize=(13, 17), dpi=200,
                         gridspec_kw={'width_ratios': [1, 1.15]})
panel_labels = ['a', 'b', 'c', 'd', 'e', 'f']

for row, res in enumerate(resolutions):
    leiden_col = f'leiden_{res}'
    categories = sample.obs[leiden_col].cat.categories
    palette = plt.cm.tab20(np.linspace(0, 1, len(categories)))
    color_map = {cat: palette[i] for i, cat in enumerate(categories)}
    colors = [color_map[c] for c in sample.obs[leiden_col]]

    ax = axes[row, 0]
    ax.scatter(umap_coords[:, 0], umap_coords[:, 1],
               c=colors, s=1.5, edgecolors='none', rasterized=True)
    for cat in categories:
        mask = sample.obs[leiden_col] == cat
        cx, cy = umap_coords[mask, 0].mean(), umap_coords[mask, 1].mean()
        ax.text(cx, cy, str(cat), fontsize=6, fontweight='bold',
                ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.12', fc='white', ec='none', alpha=0.7))
    ax.set_title(f'UMAP — Leiden {res} ({len(categories)} clusters)',
                 fontsize=10, fontweight='bold', pad=8)
    ax.axis('off')
    ax.text(-0.06, 1.05, panel_labels[row * 2], transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='top')

    ax = axes[row, 1]
    ax.scatter(sample.obs['x_centroid'], sample.obs['y_centroid'],
               c=colors, s=1.5, edgecolors='none', rasterized=True)
    ax.set_title(f'Spatial — Leiden {res} ({len(categories)} clusters)',
                 fontsize=10, fontweight='bold', pad=8)
    ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')
    handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i],
               markersize=5, label=str(cat)) for i, cat in enumerate(categories)]
    ax.legend(handles=handles, fontsize=6, frameon=False, ncol=2,
              bbox_to_anchor=(1.02, 1), loc='upper left', title='Cluster',
              title_fontsize=7, handletextpad=0.3, columnspacing=0.8)
    ax.text(-0.06, 1.05, panel_labels[row * 2 + 1], transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='top')

plt.tight_layout()
plt.savefig(FIGDIR / '(RK)SD01923BI_clustering_6panel.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(FIGDIR / '(RK)SD01923BI_clustering_6panel.pdf', bbox_inches='tight', facecolor='white')
plt.show()

FINAL_RESOLUTION = 1.5
sample.obs['leiden'] = sample.obs[f'leiden_{FINAL_RESOLUTION}'].copy()
print(f"\nFinal clustering: resolution={FINAL_RESOLUTION}, {sample.obs['leiden'].nunique()} clusters")

In [ ]:
# Save the pass-1 object, so the smear review can start from a clean read.

sample.write(PASS1_H5AD)

## Re-import post clustering

In [ ]:
# Read the pass-1 object back.

sample = sc.read(PASS1_H5AD)
sample

In [ ]:
# Per-cluster QC table sorted by median counts. Clusters at the top of this
# table are the smear candidates: many cells, few transcripts, few genes.

# Per-cluster QC summary using the log1p_norm layer for counts
cluster_qc = sample.obs.groupby('leiden_1.0').agg(
    n_cells=('total_counts', 'size'),
    median_counts=('total_counts', 'median'),
    median_genes=('n_genes_by_counts', 'median'),
    pct_low_count=('total_counts', lambda x: (x < 10).mean() * 100),
).sort_values('median_counts')

cluster_qc['pct_of_total'] = cluster_qc['n_cells'] / cluster_qc['n_cells'].sum() * 100

print(cluster_qc.to_string())

## Plotting possible smear

### Correction 2026-08-17 — smear removal by domain verdict, not by Leiden id

The two cells below shipped with `suspect = SMEAR_CLUSTERS = ['4', '5']`, copied
verbatim from SD03522_BG into all 20 notebooks by the generator. Leiden ids are
not stable across samples, so those ids mean nothing here — and even on their
home sample clusters 4 and 5 are not the lowest-count clusters (six others are)
yet removing them deletes 19.9% of cells.

Smear is a property of a spatial **territory**, not of a transcriptional cluster:
astrocytes and oligodendrocytes are dispersed across the whole section by
design, so any spatial-coherence test scores a real dispersed cell type as
"incoherent". Porting the two-gate rule down to Leiden clusters flagged 44 of 46
clusters in SD01015_BG for deletion.

The removal therefore uses the lab's validated, human-reviewed **Novae-domain**
verdicts (`dualpass_qc/extracted/tables/<SAMPLE>__decisions.csv`, joined on
`novae_domains_n6`; cohort KEEP 77 / REVIEW 41 / REMOVE 22). The Leiden-level
table is still computed below, as a **ranked diagnostic for review only** — it
deletes nothing.


In [ ]:
# ── SOURCE FIX 2026-08-17 — smear removal by validated Novae-domain verdict ──
import sys as _sys
if "/home/rodrigok/SLURM_jobs/Spatial/dualpass_rerun" not in _sys.path:
    _sys.path.insert(0, "/home/rodrigok/SLURM_jobs/Spatial/dualpass_rerun")
import json as _json1, os as _os1
import pandas as _pd
import scanpy as _sc
import smear_gate

DOMAIN_KEY_SMEAR = 'novae_domains_n6'      # primary resolution for these verdicts
DECISIONS_CSV = ("/oak/stanford/scg/lab_mpsnyder/johnck/Projects/RK/Spatial/Novae_persample_INDEPENDENT_FINAL/dualpass_qc/"
                 "extracted/tables/SD01923_BI__decisions.csv")

try:
    __TRACE__
except NameError:
    __TRACE__ = {}

if _os1.path.exists(DECISIONS_CSV) and DOMAIN_KEY_SMEAR in sample.obs.columns:
    _dec = _pd.read_csv(DECISIONS_CSV)
    _verdict = dict(zip(_dec['domain'].astype(str), _dec['verdict'].astype(str)))
    REMOVE_DOMAINS = sorted([d for d, v in _verdict.items() if v == 'REMOVE'])
    REVIEW_DOMAINS = sorted([d for d, v in _verdict.items() if v == 'REVIEW'])
    _mask = sample.obs[DOMAIN_KEY_SMEAR].astype(str).isin(REMOVE_DOMAINS).values
    print(_dec[['domain', 'n_cells', 'pct_cells', 'verdict', 'smear_score',
                'dominant_celltype', 'reason']].to_string(index=False))
    print()
    print(f"domain REMOVE: {REMOVE_DOMAINS}  -> {_mask.sum():,} cells")
    print(f"domain REVIEW: {REVIEW_DOMAINS}  (retained)")
    _n0 = sample.n_obs
    sample = sample[~_mask].copy()
    print(f"after domain smear removal: {_n0:,} -> {sample.n_obs:,}")
    __TRACE__['domain_decisions'] = _json1.loads(_dec.to_json(orient='records'))
    __TRACE__['domain_remove'] = REMOVE_DOMAINS
    __TRACE__['domain_review'] = REVIEW_DOMAINS
    __TRACE__['n_removed_by_domain'] = int(_mask.sum())
    __TRACE__['n_before_domain_removal'] = int(_n0)
else:
    raise FileNotFoundError(
        f"no domain verdicts at {DECISIONS_CSV} (or {DOMAIN_KEY_SMEAR} missing "
        f"from obs) -- refusing to fall back to hardcoded Leiden ids")

# Leiden-level table: ranked DIAGNOSTIC only. Nothing below deletes on it.
_sc.tl.rank_genes_groups(sample, 'leiden_1.0', method='wilcoxon',
                         key_added='rgg_leiden10', use_raw=False)
SMEAR_TABLE = smear_gate.cluster_gate_table(sample, 'leiden_1.0', 'rgg_leiden10')
SMEAR_TABLE = SMEAR_TABLE.sort_values('smear_rank_score', ascending=False)
print()
print("Leiden-level smear CANDIDATES (diagnostic; review against the maps):")
print(SMEAR_TABLE[['n_cells', 'pct_of_total', 'median_counts', 'count_ratio',
                   'n_strong_markers', 'largest_cc_frac', 'pct_MN',
                   'smear_rank_score', 'flag']].head(10).to_string())

DATA_DRIVEN_SMEAR = []          # removal already happened, by domain verdict
REVIEW_CLUSTERS = SMEAR_TABLE.index[SMEAR_TABLE.flag == 'CANDIDATE'].tolist()

_legacy = [c for c in ['4', '5'] if c in SMEAR_TABLE.index]
_legacy_n = int(SMEAR_TABLE.loc[_legacy, 'n_cells'].sum()) if _legacy else 0
print(f"\nlegacy ['4','5'] present={_legacy} -> would have dropped {_legacy_n:,} cells")

__TRACE__['smear_table'] = _json1.loads(SMEAR_TABLE.to_json(orient='records'))
__TRACE__['leiden_candidates'] = REVIEW_CLUSTERS
__TRACE__['legacy_clusters'] = _legacy
__TRACE__['legacy_would_drop_n'] = _legacy_n


In [ ]:
# Map the suspect clusters onto tissue. A real cell type sits where its
# anatomy sits. A smear sits in a band, an edge or a stripe.
# CHECK THIS PER SAMPLE. The cluster ids below were carried over from the
# reference notebook and almost certainly mean something else here.
suspect = (DATA_DRIVEN_SMEAR + REVIEW_CLUSTERS) or SMEAR_TABLE.index[:2].tolist()   # ranked candidates, not hardcoded ids

fig, axes = plt.subplots(1, len(suspect)+1, figsize=(16, 4), dpi=150)

axes[0].scatter(sample.obs['x_centroid'], sample.obs['y_centroid'],
                c='#dddddd', s=0.5, rasterized=True)
axes[0].set_title('All cells', fontsize=9); axes[0].set_aspect('equal')
axes[0].invert_yaxis(); axes[0].axis('off')

for ax, cl in zip(axes[1:], suspect):
    mask = sample.obs['leiden_1.0'] == cl
    axes[0].scatter(sample.obs.loc[mask, 'x_centroid'], sample.obs.loc[mask, 'y_centroid'],
                    s=0.5, rasterized=True, label=cl)
    ax.scatter(sample.obs.loc[~mask, 'x_centroid'], sample.obs.loc[~mask, 'y_centroid'],
               c='#eeeeee', s=0.5, rasterized=True)
    ax.scatter(sample.obs.loc[mask, 'x_centroid'], sample.obs.loc[mask, 'y_centroid'],
               c='#d62728', s=1, rasterized=True)
    n = mask.sum()
    med = sample.obs.loc[mask, 'total_counts'].median()
    genes = sample.obs.loc[mask, 'n_genes_by_counts'].median()
    ax.set_title(f'Cluster {cl}\nn={n:,} | med Tx={med:.0f} | med genes={genes:.0f}',
                 fontsize=8)
    ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')

plt.tight_layout()
plt.savefig(FIGDIR / '(RK)SD01923BI_suspect_clusters.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(FIGDIR / '(RK)SD01923BI_suspect_clusters.pdf', bbox_inches='tight', facecolor='white')
plt.show()

## Removing smear/debris

In [ ]:
# Drop the smear clusters and redraw.
# CHECK THIS PER SAMPLE before running, same reason as the cell above.
SMEAR_CLUSTERS = DATA_DRIVEN_SMEAR   # empty: removal is by domain verdict

n_before = sample.n_obs
sample = sample[~sample.obs['leiden_1.0'].isin(SMEAR_CLUSTERS)].copy()
print(f"Before smear filter : {n_before:,}")
print(f"After smear filter  : {sample.n_obs:,}  (removed {n_before - sample.n_obs:,} cells)")

fig, ax = plt.subplots(figsize=(7, 6), dpi=200)
ax.scatter(sample.obs['x_centroid'], sample.obs['y_centroid'], s=2, alpha=0.5,
           c='#1A56DB', edgecolors='none', rasterized=True)
ax.set_title(f'After smear cluster removal (n={sample.n_obs:,})',
             fontsize=10, fontweight='bold', pad=8)
ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')
plt.tight_layout()
plt.savefig(FIGDIR / '(RK)SD01923BI_post_smear_filter.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(FIGDIR / '(RK)SD01923BI_post_smear_filter.pdf',
            bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Numbers for the methods section: what survived, and what fraction of the
# starting cells that is.

print(f"=== Post-filtering QC summary (SD01923_BI) ===")
print(f"Cells after all filters: {sample.n_obs:,}")
print(f"Genes in panel:          {sample.n_vars:,}")
print(f"Median transcripts/cell: {sample.obs['total_counts'].median():.0f}")
print(f"Median genes/cell:       {sample.obs['n_genes_by_counts'].median():.0f}")
print(f"Cells with < 10 Tx:      {(sample.obs['total_counts'] < 10).sum():,} ({(sample.obs['total_counts'] < 10).mean()*100:.1f}%)")
print(f"Cells with < 10 genes:   {(sample.obs['n_genes_by_counts'] < 10).sum():,} ({(sample.obs['n_genes_by_counts'] < 10).mean()*100:.1f}%)")
print(f"\nFiltering removed {adata.n_obs - sample.n_obs:,} cells ({(adata.n_obs - sample.n_obs)/adata.n_obs*100:.1f}%)")

## PASS 2 after all filters

In [ ]:
# Pass 2. Renormalise from the untouched counts layer now that the debris
# is gone, then redo PCA, neighbours and UMAP. Pass 1 embeddings were
# pulled towards the smear, which is the whole point of doing this twice.

# ── Pass 2: re-normalise from raw counts after filtering ──
sample.X = sample.layers['counts'].copy()
sc.pp.normalize_total(sample, target_sum=100)
sc.pp.log1p(sample)
sample.layers['log1p_norm'] = sample.X.copy()

# ── Scale + PCA (all genes, no HVG — Salas) ──
sc.pp.scale(sample)
sc.tl.pca(sample, n_comps=50, svd_solver='arpack')

variance_ratio = sample.uns['pca']['variance_ratio']
cumulative = np.cumsum(variance_ratio)
xy = np.column_stack([np.arange(50), variance_ratio])
d = xy[-1] - xy[0]
d_norm = d / np.linalg.norm(d)
dists = np.abs(np.cross(d_norm, xy - xy[0]))
ELBOW_PC = int(np.argmax(dists)) + 1
N_PCS = 20
print(f"Geometric elbow: PC{ELBOW_PC} | Selected: PC{N_PCS}")

# ---- PCA variance plots ----
fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=200)

ax = axes[0]
ax.plot(range(1, 51), variance_ratio, 'o', color='#1A56DB', markersize=3.5,
        markeredgecolor='none', zorder=2)
ax.plot(range(1, 51), variance_ratio, '-', color='#1A56DB', linewidth=0.8,
        alpha=0.4, zorder=1)
ax.axvline(ELBOW_PC, color='#999999', linestyle=':', linewidth=0.8,
           label=f'Geometric elbow (PC{ELBOW_PC})', zorder=3)
ax.axvline(N_PCS, color='#C53030', linestyle='--', linewidth=0.8,
           label=f'Selected (PC{N_PCS})', zorder=3)
ax.fill_between(range(1, N_PCS + 1), variance_ratio[:N_PCS],
                color='#1A56DB', alpha=0.08, zorder=0)
ax.set_yscale('log')
ax.set_xlabel('Principal component', fontsize=9)
ax.set_ylabel('Variance ratio (log)', fontsize=9)
ax.set_title('Scree plot (pass 2)', fontsize=10, fontweight='bold', pad=10)
ax.legend(fontsize=7, frameon=False)
ax.tick_params(labelsize=7, length=2, width=0.5)
for s in ['top', 'right']:   ax.spines[s].set_visible(False)
for s in ['bottom', 'left']: ax.spines[s].set_linewidth(0.5)
ax.text(-0.1, 1.05, 'a', transform=ax.transAxes, fontsize=14, fontweight='bold', va='top')

ax = axes[1]
ax.plot(range(1, 51), cumulative * 100, '-', color='#276749', linewidth=1.5)
ax.axvline(ELBOW_PC, color='#999999', linestyle=':', linewidth=0.8,
           label=f'Elbow PC{ELBOW_PC}: {cumulative[ELBOW_PC-1]*100:.1f}%')
ax.axvline(N_PCS, color='#C53030', linestyle='--', linewidth=0.8,
           label=f'PC{N_PCS}: {cumulative[N_PCS-1]*100:.1f}%')
ax.set_xlabel('Principal component', fontsize=9)
ax.set_ylabel('Cumulative variance (%)', fontsize=9)
ax.set_title('Cumulative variance (pass 2)', fontsize=10, fontweight='bold', pad=10)
ax.legend(fontsize=7, frameon=False)
ax.tick_params(labelsize=7, length=2, width=0.5)
for s in ['top', 'right']:   ax.spines[s].set_visible(False)
for s in ['bottom', 'left']: ax.spines[s].set_linewidth(0.5)
ax.text(-0.1, 1.05, 'b', transform=ax.transAxes, fontsize=14, fontweight='bold', va='top')

plt.tight_layout()
plt.savefig(FIGDIR / '(RK)SD01923BI_pass2_pca_variance.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(FIGDIR / '(RK)SD01923BI_pass2_pca_variance.pdf', bbox_inches='tight', facecolor='white')
plt.show()

# ── Neighbours + UMAP (Salas: 16 neighbours) ──
sc.pp.neighbors(sample, n_neighbors=16, n_pcs=N_PCS)
sc.tl.umap(sample, min_dist=0.3)

# ── Leiden sweep ──
resolutions = [1.0, 1.2, 1.5]
for res in resolutions:
    sc.tl.leiden(sample, resolution=res, random_state=42, key_added=f'leiden_{res}')
    print(f"Resolution {res}: {sample.obs[f'leiden_{res}'].nunique()} clusters")

# ---- 6-panel clustering: UMAP + Spatial × 3 resolutions ----
umap_coords = sample.obsm['X_umap']

fig, axes = plt.subplots(3, 2, figsize=(13, 17), dpi=200,
                         gridspec_kw={'width_ratios': [1, 1.15]})
panel_labels = ['a', 'b', 'c', 'd', 'e', 'f']

for row, res in enumerate(resolutions):
    leiden_col = f'leiden_{res}'
    categories = sample.obs[leiden_col].cat.categories
    palette = plt.cm.tab20(np.linspace(0, 1, len(categories)))
    color_map = {cat: palette[i] for i, cat in enumerate(categories)}
    colors = [color_map[c] for c in sample.obs[leiden_col]]

    ax = axes[row, 0]
    ax.scatter(umap_coords[:, 0], umap_coords[:, 1],
               c=colors, s=1.5, edgecolors='none', rasterized=True)
    for cat in categories:
        mask = sample.obs[leiden_col] == cat
        cx, cy = umap_coords[mask, 0].mean(), umap_coords[mask, 1].mean()
        ax.text(cx, cy, str(cat), fontsize=6, fontweight='bold',
                ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.12', fc='white', ec='none', alpha=0.7))
    ax.set_title(f'UMAP — Leiden {res} ({len(categories)} clusters)',
                 fontsize=10, fontweight='bold', pad=8)
    ax.axis('off')
    ax.text(-0.06, 1.05, panel_labels[row * 2], transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='top')

    ax = axes[row, 1]
    ax.scatter(sample.obs['x_centroid'], sample.obs['y_centroid'],
               c=colors, s=1.5, edgecolors='none', rasterized=True)
    ax.set_title(f'Spatial — Leiden {res} ({len(categories)} clusters)',
                 fontsize=10, fontweight='bold', pad=8)
    ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')
    handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i],
               markersize=5, label=str(cat)) for i, cat in enumerate(categories)]
    ax.legend(handles=handles, fontsize=6, frameon=False, ncol=2,
              bbox_to_anchor=(1.02, 1), loc='upper left', title='Cluster',
              title_fontsize=7, handletextpad=0.3, columnspacing=0.8)
    ax.text(-0.06, 1.05, panel_labels[row * 2 + 1], transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='top')

plt.tight_layout()
plt.savefig(FIGDIR / '(RK)SD01923BI_pass2_clustering_6panel.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(FIGDIR / '(RK)SD01923BI_pass2_clustering_6panel.pdf', bbox_inches='tight', facecolor='white')
plt.show()

FINAL_RESOLUTION = 1.0
sample.obs['leiden'] = sample.obs[f'leiden_{FINAL_RESOLUTION}'].copy()
print(f"\nFinal: resolution={FINAL_RESOLUTION}, {sample.obs['leiden'].nunique()} clusters")

# ── Restore log-norm for downstream ──
sample.X = sample.layers['log1p_norm'].copy()

def get_lognorm(gene):
    idx = sample.var_names.get_loc(gene)
    layer = sample.layers['log1p_norm']
    return (np.asarray(layer[:, idx].todense()).flatten()
            if sp.issparse(layer) else layer[:, idx].flatten())

## Manual markers - first pass

In [ ]:
# Check which canonical markers actually exist in the panel before scoring
# anything. A cell type with one surviving marker cannot be scored.

markers = {
    'Motor neurons':        ['CHAT', 'SLC5A7', 'STMN2', 'ISL1', 'MNX1', 'PRPH'],
    'Excitatory neurons':   ['SLC17A6', 'RBFOX3', 'STMN2', 'NRGN'],
    'Inhibitory neurons':   ['GAD1', 'GAD2', 'SLC32A1', 'RBFOX3'],
    'Oligodendrocytes':     ['MOG', 'MBP', 'PLP1', 'MAG', 'MOBP'],
    'OPCs':                 ['OLIG2', 'PDGFRA', 'CSPG4', 'VCAN'],
    'Astrocytes':           ['GFAP', 'AQP4', 'SLC1A2', 'SLC1A3', 'ALDH1L1'],
    'Schwann cells':        ['MPZ', 'PMP22', 'PRX', 'SOX10'],
    'Microglia':            ['TREM2', 'CD68', 'CX3CR1', 'P2RY12', 'CSF1R', 'AIF1'],
    'T cells':              ['CD4', 'CD3E', 'CD3D', 'CD8A'],
    'Macrophages':          ['CD68', 'CD163', 'MRC1', 'MSR1'],
    'Endothelial':          ['CLDN5', 'PECAM1', 'FLT1', 'VWF'],
    'Pericytes':            ['PDGFRB', 'RGS5', 'NOTCH3', 'KCNJ8'],
    'Ependymal':            ['FOXJ1', 'DNAH11', 'CFAP299'],
    'Meningeal/Fibroblasts':['DCN', 'COL1A1', 'COL1A2', 'LUM'],
}

for ct, genes in markers.items():
    present = [g for g in genes if g in sample.var_names]
    missing = [g for g in genes if g not in sample.var_names]
    print(f"{'✓' if len(present)>=2 else '⚠' if len(present)==1 else '✗'} {ct:25s} | present({len(present)}): {present}  | missing: {missing}")

In [ ]:
# Score each cell type and assign every cell to its highest-scoring one.
# This is a rough first pass, not a final annotation. Types with a single
# marker are printed separately and need a manual call.

sample.X = sample.layers['log1p_norm'].copy()

scored = {}
single_marker = {}

for cell_type, genes in markers.items():
    available = [g for g in genes if g in sample.var_names]
    if len(available) >= 2:
        sc.tl.score_genes(sample, gene_list=available, score_name=f'score_{cell_type}')
        scored[cell_type] = available
        print(f"✓ {cell_type}: scored with {available}")
    elif len(available) == 1:
        single_marker[cell_type] = available[0]
        print(f"⚠ {cell_type}: single marker {available[0]} — not scored, use directly")
    else:
        print(f"✗ {cell_type}: no markers")

score_cols = [c for c in sample.obs.columns if c.startswith('score_')]
score_matrix = sample.obs[score_cols].copy()
score_matrix.columns = [c.replace('score_', '') for c in score_cols]

sample.obs['cell_type'] = score_matrix.idxmax(axis=1)
sample.obs['cell_type_score'] = score_matrix.max(axis=1)

print(f"\n=== Cell type distribution (score-based, {len(scored)} types) ===")
print(sample.obs['cell_type'].value_counts())
print(f"\n⚠ Single-marker types (annotate manually from expression):")
for ct, gene in single_marker.items():
    n_pos = (sample[:, gene].X.toarray().flatten() > 0).sum() if sp.issparse(sample.X) else (sample[:, gene].X.flatten() > 0).sum()
    print(f"  {ct}: {gene} — {n_pos:,} positive cells")

## DEGs per assignment

In [ ]:
# Wilcoxon markers per assigned type. If the dotplot does not recover the
# genes that were scored, the assignment is not holding up.

sample.X = sample.layers['log1p_norm'].copy()

sc.tl.rank_genes_groups(
    sample, groupby='cell_type', method='wilcoxon',
    key_added='rank_genes_cell_type', pts=True, use_raw=False
)

sc.pl.rank_genes_groups_dotplot(
    sample, key='rank_genes_cell_type', n_genes=5, values_to_plot='logfoldchanges',
    cmap='bwr', min_logfoldchange=1.0, groupby='cell_type',
    save='_SD01923BI_de_top5_per_celltype.png'
)

## SVGs

In [ ]:
# Moran's I on the first 200 genes for a quick spatially variable gene
# list. Wrapped in a try block because it is a nice-to-have, not a gate.

try:
    sq.gr.spatial_neighbors(sample, coord_type='generic', n_rings=1, key_added='spatial_moran')
    sq.gr.spatial_autocorr(
        sample, mode='moran', n_perms=100, n_jobs=1,
        genes=sample.var_names[:200],
        connectivity_key='spatial_moran_connectivities'
    )
    sv_genes = sample.uns['moranI'].sort_values('I', ascending=False).head(20)
    print("Top spatially variable genes (Moran's I):")
    print(sv_genes[['I', 'pval_sim', 'pval_norm']])

    top_sv = sv_genes.index[0]
    expr = np.asarray(sample[:, top_sv].X.todense()).flatten() if sp.issparse(sample.X) else sample[:, top_sv].X.flatten()

    fig, ax = plt.subplots(figsize=(7, 6))
    sc_plot = ax.scatter(sample.obs['x_centroid'], sample.obs['y_centroid'],
                         c=expr, s=0.3, cmap='viridis', rasterized=True)
    plt.colorbar(sc_plot, ax=ax)
    ax.set_title(f"{top_sv} — top spatially variable gene (Moran's I)")
    ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')
    plt.tight_layout()
    plt.savefig(FIGDIR / f'(RK)SD01923BI_{top_sv}_spatial.png', dpi=150, bbox_inches='tight')
    plt.savefig(FIGDIR / f'(RK)SD01923BI_{top_sv}_spatial.pdf', bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f"Moran's I skipped: {e}")

In [ ]:
# Object summary after pass 2.

sample

In [ ]:
# Save the pass-2 object.

sample.write(PASS2_H5AD)

In [ ]:
# Cell types on UMAP and on tissue. The tissue panel is the one that
# matters here, since the annotation is only believable if the types land
# where the anatomy says they should.

cell_types = sample.obs['cell_type'].cat.categories if hasattr(sample.obs['cell_type'], 'cat') else sample.obs['cell_type'].unique()
n_types = len(cell_types)
palette = dict(zip(cell_types, plt.cm.tab20.colors[:n_types]))

fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=200,
                         gridspec_kw={'width_ratios': [1, 1.15]})

# UMAP
ax = axes[0]
for ct in cell_types:
    mask = sample.obs['cell_type'] == ct
    ax.scatter(sample.obsm['X_umap'][mask, 0], sample.obsm['X_umap'][mask, 1],
               c=[palette[ct]], s=0.5, label=ct, edgecolors='none', rasterized=True)
ax.set_title('UMAP — cell types', fontsize=10, fontweight='bold')
ax.axis('off')
ax.legend(fontsize=7, frameon=False, markerscale=5, bbox_to_anchor=(0.01, 0.01),
          loc='lower left', ncol=2)

# Spatial
ax = axes[1]
for ct in cell_types:
    mask = sample.obs['cell_type'] == ct
    ax.scatter(sample.obs.loc[mask, 'x_centroid'], sample.obs.loc[mask, 'y_centroid'],
               c=[palette[ct]], s=0.5, label=ct, edgecolors='none', rasterized=True)
ax.set_title('Spatial — cell types', fontsize=10, fontweight='bold')
ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')
ax.legend(fontsize=7, frameon=False, markerscale=5, bbox_to_anchor=(1.02, 1),
          loc='upper left', ncol=1)

plt.tight_layout()
plt.savefig(FIGDIR / '(RK)SD01923BI_celltype_spatial.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(FIGDIR / '(RK)SD01923BI_celltype_spatial.pdf', bbox_inches='tight', facecolor='white')
plt.show()

print(sample.obs['cell_type'].value_counts())

### QC decision trace

Written to JSON so QC severity can be tested against disease group and spinal level.


In [ ]:
# ── SOURCE FIX 2026-08-17 — machine-readable QC decision trace ───────────
# Print-only traces cannot be tested against disease group or spinal level.
import json as _json2, os as _os3
import numpy as _np2

def _f(x):
    try:
        return float(x)
    except Exception:
        return None

try:
    __TRACE__
except NameError:
    __TRACE__ = {}

__TRACE__['raw_cells'] = int(adata.n_obs)
__TRACE__['n_genes_panel'] = int(adata.n_vars)
__TRACE__['final_cells'] = int(sample.n_obs)
__TRACE__['raw_median_counts'] = _f(adata.obs['total_counts'].median())
__TRACE__['raw_median_genes'] = _f(adata.obs['n_genes_by_counts'].median())
__TRACE__['final_median_counts'] = _f(sample.obs['total_counts'].median())
__TRACE__['final_median_genes'] = _f(sample.obs['n_genes_by_counts'].median())
__TRACE__['pct_lt10_tx_raw'] = _f((adata.obs['total_counts'] < 10).mean() * 100)
__TRACE__['pct_lt10_genes_raw'] = _f((adata.obs['n_genes_by_counts'] < 10).mean() * 100)
__TRACE__['neg_frac_median'] = _f(globals().get('neg_frac_median'))
__TRACE__['snr_aggregate'] = _f(globals().get('snr_aggregate'))
__TRACE__['n_after_transcript'] = int(globals().get('n_after_transcript', -1))
__TRACE__['n_after_spatial'] = int(globals().get('n_after_spatial', -1))
if 'cell_type' in sample.obs.columns:
    __TRACE__['cell_type_counts'] = {str(k): int(v) for k, v in
                                     sample.obs['cell_type'].value_counts().items()}
if 'is_MN' in sample.obs.columns:
    __TRACE__['n_MN_final'] = int(__import__('pandas').Series(sample.obs['is_MN']).astype('boolean').fillna(False).sum())
if 'is_MN' in adata.obs.columns:
    __TRACE__['n_MN_raw'] = int(__import__('pandas').Series(adata.obs['is_MN']).astype('boolean').fillna(False).sum())
for _k in ['MIN_TRANSCRIPTS', 'MIN_NEIGHBOURS', 'N_PCS', 'ELBOW_PC', 'DOMAIN_QC_KEY']:
    if _k in globals():
        __TRACE__[_k] = globals()[_k]
if 'domain_qc' in globals():
    __TRACE__['n_domains_scored'] = int(domain_qc.shape[0])
if 'flagged' in globals():
    __TRACE__['flagged_domains'] = [str(_x) for _x in flagged]


# ── VH RELABEL V6 2026-08-18 - counts for the relabelled columns ───────────
# is_MN, in_VH, in_GM, is_MN_invh and is_GADvh are nullable: <NA> for every cell
# on a section with no ventral-horn mask. np.asarray(...).astype(bool) treats
# pd.NA as True and would turn every cell of such a section into a motor neuron,
# so every count goes through astype("boolean").fillna(False).
import pandas as _pd_tr

assert globals().get("__RELABEL_V4REAL_OK__") is True, \
    "the v4REAL relabel cell did not complete -- refusing to write a trace that looks clean"


def _bool_mask(col):
    return _pd_tr.Series(col).astype("boolean").fillna(False).to_numpy(dtype=bool)


for _obj, _tag in ((adata, "raw"), (sample, "final")):
    for _c in ("is_MN", "is_MN_v6_asdelivered", "is_MN_invh", "is_MN_eligible",
               "in_VH", "in_GM", "in_VH_not_GM", "is_GAD", "is_GADvh",
               "is_gabaergic_neuron", "is_Interneuron", "is_TDP",
               "tdp_evaluated", "is_neuron_v3", "is_neuron_general",
               "is_neuron_prev", "has_vh_data", "vh_mask_empty"):
        if _c in _obj.obs.columns:
            __TRACE__[f"n_{_c}_{_tag}"] = int(_bool_mask(_obj.obs[_c]).sum())
    if "is_MN" in _obj.obs.columns:
        __TRACE__[f"n_is_MN_na_{_tag}"] = int(
            _pd_tr.Series(_obj.obs["is_MN"]).astype("boolean").isna().sum())
    for _c in ("neuron_class", "cell_type_mw", "pathologist_class_mw", "vh_source"):
        if _c in _obj.obs.columns:
            __TRACE__[f"{_c}_{_tag}"] = {
                str(_k): int(_v) for _k, _v in _obj.obs[_c].value_counts().items()}
# the stale column must be gone, not merely unused
assert "is_neuron" not in adata.obs.columns, \
    "stale obs['is_neuron'] survived the relabel -- consumers keyed on that string " \
    "would read the pre-v4 call (10,085 vs 6,035) with no error"
__TRACE__["mn_definition"] = ("parquet v4 is_MN = is_neuron_v3 & ~is_GAD & in_VH & "
                              "cell_area_um2 >= 250; is_MN_invh is the denominator "
                              "the size cut is taken from, is_MN_eligible is the "
                              "denominator that survives a missing VH mask")
__TRACE__["published_cohort_counts"] = (
    "over our 20 sections, whole-section and AFTER dedupe: 599 is_MN, 6,035 "
    "is_neuron_v3, 130 pathologist labels. Summing the V4_FACTS table gives 601 "
    "and 6,037 because 16 rows are duplicated (all of the difference on our 20 "
    "is SD04219_BI). SD01015_BG and SD01616_BI are UNDETERMINED, never 0.")

_TRACE_DIR = _os3.environ.get("DUALPASS_TRACE_DIR",
                             "/oak/stanford/scg/lab_mpsnyder/johnck/Projects/RK/Spatial/Novae_IND_QC/runs_srcfixed")
_os3.makedirs(_TRACE_DIR, exist_ok=True)
_TRACE_PATH = _os3.path.join(_TRACE_DIR, "trace_src_SD01923_BI.json")
with open(_TRACE_PATH, "w") as _fh:
    _json2.dump(__TRACE__, _fh, indent=1, default=str)
print("TRACE WRITTEN ->", _TRACE_PATH)
